In [31]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/processed/9558")

lap_data = pd.read_parquet(
    DATA_DIR / "lap_performance.parquet"
)

fuel_features = pd.read_parquet(
    DATA_DIR / "fuel_features.parquet"
)

traffic_features = pd.read_parquet(
    DATA_DIR / "traffic_features.parquet"
)

tyre_features = pd.read_parquet(
    DATA_DIR / "tyre_features.parquet"
)

print("Lap data:", lap_data.shape)
print("Fuel features:", fuel_features.shape)
print("Traffic features:", traffic_features.shape)
print("Tyre features:", tyre_features.shape)

Lap data: (962, 19)
Fuel features: (692, 3)
Traffic features: (962, 6)
Tyre features: (650, 4)


In [32]:
datasets = {
    "lap_performance": lap_data,
    "fuel_features": fuel_features,
    "traffic_features": traffic_features,
    "tyre_features": tyre_features,
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(df.columns.tolist())

print("\n===== LAP PERFORMANCE SAMPLE =====")
print(lap_data.head(2))

print("\n===== FUEL FEATURES SAMPLE =====")
print(fuel_features.head(2))

print("\n===== TRAFFIC FEATURES SAMPLE =====")
print(traffic_features.head(2))

print("\n===== TYRE FEATURES SAMPLE =====")
print(tyre_features.head(2))


lap_performance
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age']

fuel_features
['driver_number', 'lap_number', 'fuel_effect']

traffic_features
['driver_number', 'lap_number', 'position', 'traffic_ahead', 'traffic_behind', 'position_change']

tyre_features
['driver_number', 'lap_number', 'tyre_age', 'tyre_degradation_effect']

===== LAP PERFORMANCE SAMPLE =====
   meeting_key  session_key  driver_number  ...  stint_number compound  tyre_age
0         1240         9558              1  ...           1.0   MEDIUM       0.0
1         1240         9558              1  ...           1.0   MEDIUM       1.0

[2 rows x 19 columns]

===== FUEL FEATURES SAMPLE =====
   driver_number  lap_number  fuel_effect
0              1           1 

In [33]:
print("Lap performance duplicate keys:",
      lap_data.duplicated(["driver_number", "lap_number"]).sum())

print("Fuel feature duplicate keys:",
      fuel_features.duplicated(["driver_number", "lap_number"]).sum())

print("Traffic feature duplicate keys:",
      traffic_features.duplicated(["driver_number", "lap_number"]).sum())

print("Tyre feature duplicate keys:",
      tyre_features.duplicated(["driver_number", "lap_number"]).sum())

Lap performance duplicate keys: 0
Fuel feature duplicate keys: 0
Traffic feature duplicate keys: 1
Tyre feature duplicate keys: 0


In [34]:
lap_keys = lap_data[["driver_number", "lap_number"]].drop_duplicates()

fuel_keys = fuel_features[["driver_number", "lap_number"]]
traffic_keys = traffic_features[["driver_number", "lap_number"]]
tyre_keys = tyre_features[["driver_number","lap_number"]]

print("Fuel coverage:",
      lap_keys.merge(
          fuel_keys,
          on=["driver_number", "lap_number"],
          how="inner"
      ).shape[0])

print("Traffic coverage:",
      lap_keys.merge(
          traffic_keys,
          on=["driver_number","lap_number"],
          how="inner"
      ).shape[0])

print("Tyre coverage:",
      lap_keys.merge(
          tyre_keys,
          on=["driver_number", "lap_number"],
          how="inner"
      ).shape[0])

Fuel coverage: 692
Traffic coverage: 962
Tyre coverage: 650


In [35]:
traffic_features = traffic_features.drop_duplicates(
    subset=["driver_number", "lap_number"]
).copy()

print(
    "Traffic duplicate keys:",
    traffic_features.duplicated(
        ["driver_number", "lap_number"]
    ).sum()
)

print("Traffic features shape:", traffic_features.shape)

Traffic duplicate keys: 0
Traffic features shape: (961, 6)


In [36]:
enhanced_lap_data = lap_data.copy()

enhanced_lap_data = enhanced_lap_data.merge(
    fuel_features,
    on=["driver_number","lap_number"],
    how="left",
    validate="one_to_one"
)

enhanced_lap_data = enhanced_lap_data.merge(
    traffic_features,
    on=["driver_number","lap_number"],
    how="left",
    validate="one_to_one"
)

enhanced_lap_data = enhanced_lap_data.merge(
    tyre_features,
    on=["driver_number","lap_number"],
    how="left",
    validate="one_to_one"
)
print("Enhanced pace data shape:", enhanced_lap_data.shape)
print("\nColumns:")
print(enhanced_lap_data.columns.tolist())


Enhanced pace data shape: (962, 26)

Columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age_x', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_age_y', 'tyre_degradation_effect']


In [37]:
print(
    enhanced_lap_data[
        [
            "fuel_effect",
            "traffic_ahead",
            "traffic_behind",
            "position_change",
            "tyre_degradation_effect"
        ]
    ].isna().sum()
)

fuel_effect                270
traffic_ahead                1
traffic_behind               1
position_change              1
tyre_degradation_effect    312
dtype: int64


In [38]:
tyre_age_check = enhanced_lap_data[
    ["tyre_age_x", "tyre_age_y"]
].dropna()

print("Rows with both tyre ages:", len(tyre_age_check))

print("Mismatched tyre ages:",
      (
          tyre_age_check["tyre_age_x"]
          != tyre_age_check["tyre_age_y"]
      ).sum()
)

Rows with both tyre ages: 650
Mismatched tyre ages: 0


In [39]:
enhanced_lap_data = enhanced_lap_data.drop(
    columns=["tyre_age_y"]
).rename(
    columns={"tyre_age_x": "tyre_age"}
)

print(enhanced_lap_data.columns.tolist())
print("\nShape:", enhanced_lap_data.shape)

['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect']

Shape: (962, 25)


In [40]:
missing_fuel = enhanced_lap_data[
    enhanced_lap_data["fuel_effect"].isna()
]

missing_tyre = enhanced_lap_data[
    enhanced_lap_data["tyre_degradation_effect"].isna()
]

missing_traffic = enhanced_lap_data[
    enhanced_lap_data["traffic_ahead"].isna()
]

print("Missing fuel:", len(missing_fuel))
print("Missing tyre:", len(missing_tyre))
print("Missing traffic:", len(missing_traffic))

Missing fuel: 270
Missing tyre: 312
Missing traffic: 1


In [41]:
print(
    missing_traffic[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "position",
            "compound",
            "tyre_age"
        ]
    ]
)

print(
    missing_fuel[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "compound",
            "tyre_age"
        ]
    ].head(10)
)

print(
    missing_tyre[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "compound",
            "tyre_age"
        ]
    ].head(10)
)


     driver_number  lap_number  lap_duration  position compound  tyre_age
208             11           1           NaN       NaN     HARD       0.0
    driver_number  lap_number  lap_duration      compound  tyre_age
26              1          27       119.169  INTERMEDIATE       0.0
27              1          28       101.176  INTERMEDIATE       1.0
28              1          29       101.533  INTERMEDIATE       2.0
29              1          30       102.098  INTERMEDIATE       3.0
30              1          31       102.025  INTERMEDIATE       4.0
31              1          32       102.538  INTERMEDIATE       5.0
32              1          33       102.606  INTERMEDIATE       6.0
33              1          34       102.447  INTERMEDIATE       7.0
34              1          35       103.029  INTERMEDIATE       8.0
35              1          36       101.958  INTERMEDIATE       9.0
    driver_number  lap_number  lap_duration      compound  tyre_age
0               1           1       

In [42]:
print("===== COMPOUND COUNTS =====")
print(
    enhanced_lap_data["compound"]
    .value_counts(dropna=False)
)

print("\n===== PIT-OUT LAPS =====")
print(
    enhanced_lap_data["is_pit_out_lap"]
    .value_counts(dropna=False)
)

print("\n===== MISSING LAP DURATION =====")
print(
    enhanced_lap_data["lap_duration"].isna().sum()
)

print("\n===== LAP DURATION SUMMARY =====")
print(
    enhanced_lap_data["lap_duration"].describe()
)

===== COMPOUND COUNTS =====
compound
MEDIUM          508
INTERMEDIATE    234
SOFT            174
HARD             44
NaN               2
Name: count, dtype: int64

===== PIT-OUT LAPS =====
is_pit_out_lap
False    917
True      45
Name: count, dtype: int64

===== MISSING LAP DURATION =====
3

===== LAP DURATION SUMMARY =====
count    959.000000
mean      96.833269
std        7.612357
min       88.293000
25%       91.717000
50%       93.219000
75%      101.995000
max      128.459000
Name: lap_duration, dtype: float64


In [43]:
print("===== PIT-OUT COLUMN INFO =====")
print("dtype:", enhanced_lap_data["is_pit_out_lap"].dtype)

print("\nUnique values:")
print(enhanced_lap_data["is_pit_out_lap"].unique())

print("\nValue counts:")
print(enhanced_lap_data["is_pit_out_lap"].value_counts(dropna=False))

===== PIT-OUT COLUMN INFO =====
dtype: bool

Unique values:
[False  True]

Value counts:
is_pit_out_lap
False    917
True      45
Name: count, dtype: int64


In [44]:
print("Total rows:", len(enhanced_lap_data))

step1 = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
].copy()

step2 = step1[
    step1["is_pit_out_lap"] == False
].copy()

print("After removing pit out:", len(step2))

step3 = step2[
    step2["compound"].isin(["SOFT","MEDIUM","HARD"])
].copy()

print("After dry compounds:", len(step3))

step4 = step3[
    ~step3["lap_number"].isin([19,25])
].copy()

print("After removing low-grip laps:", len(step4))

print("\nFinal pit-out counts:")
print(step4["is_pit_out_lap"].value_counts())

print("\nFinal compound counts:")
print(step4["compound"].value_counts())


Total rows: 962
After removing pit out: 914
After dry compounds: 703
After removing low-grip laps: 668

Final pit-out counts:
is_pit_out_lap
False    668
Name: count, dtype: int64

Final compound counts:
compound
MEDIUM    467
SOFT      161
HARD       40
Name: count, dtype: int64


In [45]:
pace_model_data = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
    & (enhanced_lap_data["is_pit_out_lap"] == False)
    & enhanced_lap_data["compound"].isin(["SOFT", "MEDIUM", "HARD"])
    & (~enhanced_lap_data["lap_number"].isin([19, 25]))
].copy()

print("===== PACE MODEL DATA =====")
print("Shape:", pace_model_data.shape)

print("\n===== COMPOUNDS =====")
print(pace_model_data["compound"].value_counts())

print("\n===== PIT-OUT LAPS =====")
print(pace_model_data["is_pit_out_lap"].value_counts())

print("\n===== MISSING LAP DURATION =====")
print(pace_model_data["lap_duration"].isna().sum())

===== PACE MODEL DATA =====
Shape: (668, 25)

===== COMPOUNDS =====
compound
MEDIUM    467
SOFT      161
HARD       40
Name: count, dtype: int64

===== PIT-OUT LAPS =====
is_pit_out_lap
False    668
Name: count, dtype: int64

===== MISSING LAP DURATION =====
0


In [46]:
feature_columns = [
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect"
]

for column in feature_columns:
    missing = pace_model_data[column].isna().sum()
    available = pace_model_data[column].notna().sum()
    total = len(pace_model_data)

    print(
        f"{column}: "
        f"{available}/{total} available "
        f"({available / total * 100:.1f}%), "
        f"{missing} missing"
    )

print(
    pace_model_data[
        pace_model_data["fuel_effect"].isna()
    ][
        ["driver_number", "lap_number", "compound", "tyre_age", "lap_duration"]
    ].to_string(index=False)
)

print(
    pace_model_data[
        pace_model_data["tyre_degradation_effect"].isna()
    ][
        ["driver_number", "lap_number", "compound", "tyre_age", "lap_duration"]
    ].to_string(index=False)
)

fuel_effect: 657/668 available (98.4%), 11 missing
traffic_ahead: 668/668 available (100.0%), 0 missing
traffic_behind: 668/668 available (100.0%), 0 missing
position_change: 668/668 available (100.0%), 0 missing
tyre_degradation_effect: 650/668 available (97.3%), 18 missing
 driver_number  lap_number compound  tyre_age  lap_duration
             2          27   MEDIUM      26.0       112.246
             4          27   MEDIUM      26.0       104.806
            14          27   MEDIUM      26.0       108.513
            20          27   MEDIUM      26.0       111.325
            22          27   MEDIUM      26.0       108.126
            23          27   MEDIUM      26.0       107.754
            31          26   MEDIUM       4.0       118.710
            44          27   MEDIUM      26.0       105.183
            63          27   MEDIUM      26.0       107.698
            81          27   MEDIUM      26.0       109.747
            81          28   MEDIUM      27.0       112.976
 dri

In [47]:
fuel_coefficient = -0.084122

missing_fuel = pace_model_data["fuel_effect"].isna()

pace_model_data.loc[missing_fuel,"fuel_effect"] = (
    (pace_model_data.loc[missing_fuel, "lap_number"] - 1)*fuel_coefficient
)

print("Missing fuel_effect after filling:",
    pace_model_data["fuel_effect"].isna().sum())

print("Filled fuel effects:")

print(
    pace_model_data.loc[
        missing_fuel,
        ["driver_number", "lap_number", "fuel_effect"]
    ].to_string(index=False)
)

Missing fuel_effect after filling: 0
Filled fuel effects:
 driver_number  lap_number  fuel_effect
             2          27    -2.187172
             4          27    -2.187172
            14          27    -2.187172
            20          27    -2.187172
            22          27    -2.187172
            23          27    -2.187172
            31          26    -2.103050
            44          27    -2.187172
            63          27    -2.187172
            81          27    -2.187172
            81          28    -2.271294


In [48]:
missing_tyre = pace_model_data["tyre_degradation_effect"].isna()

pace_model_data.loc[missing_tyre, "tyre_degradation_effect"] = 0.0

print("Missing tyre degradation after filling:",
      pace_model_data["tyre_degradation_effect"].isna().sum())

print("Final feature coverage:")

print(
    pace_model_data[
        [
            "lap_duration",
            "fuel_effect",
            "traffic_ahead",
            "traffic_behind",
            "position_change",
            "tyre_degradation_effect"
        ]
    ].isna().sum()
)

Missing tyre degradation after filling: 0
Final feature coverage:
lap_duration               0
fuel_effect                0
traffic_ahead              0
traffic_behind             0
position_change            0
tyre_degradation_effect    0
dtype: int64


In [49]:
print("===== ENHANCED PACE DATA SUMMARY =====")

print("\nLap duration:")
print(pace_model_data["lap_duration"].describe())

print("\nFuel effect:")
print(pace_model_data["fuel_effect"].describe())

print("\nTyre age:")
print(pace_model_data["tyre_age"].describe())

print("\nTyre degradation effect:")
print(pace_model_data["tyre_degradation_effect"].describe())

print("\nTraffic ahead:")
print(pace_model_data["traffic_ahead"].describe())

print("\nTraffic behind:")
print(pace_model_data["traffic_behind"].describe())

print("\nPosition change:")
print(pace_model_data["position_change"].describe())

===== ENHANCED PACE DATA SUMMARY =====

Lap duration:
count    668.000000
mean      93.058708
std        3.567319
min       88.293000
25%       91.140000
50%       92.444500
75%       93.480250
max      118.710000
Name: lap_duration, dtype: float64

Fuel effect:
count    668.000000
mean      -1.917301
std        1.418689
min       -4.290221
25%       -3.533123
50%       -1.430074
75%       -0.672976
max       -0.000000
Name: fuel_effect, dtype: float64

Tyre age:
count    668.000000
mean       9.805389
std        6.804917
min        0.000000
25%        4.000000
50%        9.000000
75%       14.000000
max       27.000000
Name: tyre_age, dtype: float64

Tyre degradation effect:
count    668.000000
mean       2.110555
std        1.464720
min        0.000000
25%        0.860978
50%        1.937200
75%        3.013422
max        5.811599
Name: tyre_degradation_effect, dtype: float64

Traffic ahead:
count    668.000000
mean       0.944611
std        0.228910
min        0.000000
25%        1.

In [104]:
import statsmodels.formula.api as smf

baseline_model = smf.ols(
    "lap_duration ~ C(driver_number) + C(compound) + tyre_age",
    data=pace_model_data
).fit()

print(baseline_model.summary())

                            OLS Regression Results                            
Dep. Variable:           lap_duration   R-squared:                       0.264
Model:                            OLS   Adj. R-squared:                  0.240
Method:                 Least Squares   F-statistic:                     11.02
Date:                Thu, 01 Oct 2026   Prob (F-statistic):           2.70e-31
Time:                        15:24:23   Log-Likelihood:                -1694.6
No. Observations:                 668   AIC:                             3433.
Df Residuals:                     646   BIC:                             3532.
Df Model:                          21                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 88

In [51]:
import statsmodels.formula.api as smf
enhanced_model = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + tyre_age
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    """,
    data=pace_model_data
).fit()

print(enhanced_model.summary())

                            OLS Regression Results                            
Dep. Variable:           lap_duration   R-squared:                       0.339
Model:                            OLS   Adj. R-squared:                  0.313
Method:                 Least Squares   F-statistic:                     13.16
Date:                Fri, 02 Oct 2026   Prob (F-statistic):           5.47e-43
Time:                        11:06:30   Log-Likelihood:                -1658.8
No. Observations:                 668   AIC:                             3370.
Df Residuals:                     642   BIC:                             3487.
Df Model:                          25                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\2247647040.py:15: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit()


In [52]:
train_data = pace_model_data[
    pace_model_data["lap_number"] < 45
].copy()

test_data = pace_model_data[
    pace_model_data["lap_number"] >= 45
].copy()

print("Training rows:", len(train_data))
print("Test rows:", len(test_data))

Training rows: 535
Test rows: 133


In [53]:
enhanced_model_train = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + tyre_age
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    """,
    data=train_data
).fit()

test_predictions = enhanced_model_train.predict(test_data)

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

test_r2 = r2_score(
    test_data["lap_duration"],
    test_predictions
)

test_rmse = np.sqrt(
    mean_squared_error(
        test_data["lap_duration"],
        test_predictions
    )
)

print("===== ENHANCED MODEL TEST PERFORMANCE =====")
print("Test R²:", test_r2)
print("Test RMSE:", test_rmse)

===== ENHANCED MODEL TEST PERFORMANCE =====
Test R²: -0.598773123352653
Test RMSE: 1.6408130015000804


C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\2439155361.py:14: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit()


In [54]:
enhanced_model_v2 = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    """,
    data=train_data
).fit()

test_predictions_v2 = enhanced_model_v2.predict(test_data)

test_r2_v2 = r2_score(
    test_data["lap_duration"],
    test_predictions_v2
)

test_rmse_v2 = np.sqrt(
    mean_squared_error(
        test_data["lap_duration"],
        test_predictions_v2
    )
)

print("===== ENHANCED MODEL V2 =====")
print("Test R²:", test_r2_v2)
print("Test RMSE:", test_rmse_v2)

===== ENHANCED MODEL V2 =====
Test R²: -0.5987731233534486
Test RMSE: 1.6408130015004887


In [55]:
baseline_model_train = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + tyre_age
    """,
    data=train_data
).fit()

baseline_predictions = baseline_model_train.predict(test_data)

baseline_r2 = r2_score(
    test_data["lap_duration"],
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        test_data["lap_duration"],
        baseline_predictions
    )
)

print("===== BASELINE TEST PERFORMANCE =====")
print("Test R²:", baseline_r2)
print("Test RMSE:", baseline_rmse)

===== BASELINE TEST PERFORMANCE =====
Test R²: -2.8653388837679707
Test RMSE: 2.551287827797025


In [56]:
print("===== TEST LAP DISTRIBUTION =====")

print(
    test_data["lap_number"]
    .value_counts()
    .sort_index()
)

print("\n===== TEST LAP DURATION BY LAP =====")

print(
    test_data.groupby("lap_number")["lap_duration"]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

===== TEST LAP DISTRIBUTION =====
lap_number
45    18
46    18
47    18
48    17
49    18
50    18
51    14
52    12
Name: count, dtype: int64

===== TEST LAP DURATION BY LAP =====
            count    mean     min     max
lap_number                               
45             18  90.447  89.137  93.159
46             18  90.356  89.233  91.688
47             18  90.469  89.049  93.599
48             17  90.749  88.952  95.931
49             18  91.055  89.283  97.360
50             18  90.766  89.286  94.150
51             14  90.950  88.748  94.814
52             12  90.548  88.293  92.090


In [57]:
comparison = test_data[
    [
        "driver_number",
        "lap_number",
        "compound",
        "tyre_age",
        "lap_duration"
    ]
].copy()

comparison["predicted_lap_duration"] = test_predictions_v2

comparison["error"] = (
    comparison["predicted_lap_duration"]
    - comparison["lap_duration"]
)

print("===== PREDICTION COMPARISON =====")

print(
    comparison[
        [
            "driver_number",
            "lap_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "predicted_lap_duration",
            "error"
        ]
    ]
    .sort_values(["lap_number", "driver_number"])
    .head(30)
    .round(3)
    .to_string(index=False)
)

print("\n===== ERROR SUMMARY =====")

print(
    comparison["error"].describe().round(3)
)

===== PREDICTION COMPARISON =====
 driver_number  lap_number compound  tyre_age  lap_duration  predicted_lap_duration  error
             1          45     HARD       6.0        89.137                  89.967  0.830
             2          45     SOFT       6.0        90.613                  90.693  0.080
             3          45     SOFT       7.0        90.955                  91.706  0.751
             4          45     SOFT       5.0        89.635                  89.635  0.000
            11          45   MEDIUM       7.0        90.565                  91.823  1.258
            14          45   MEDIUM       6.0        89.871                  91.598  1.727
            16          45     SOFT       7.0        89.911                  90.826  0.915
            18          45   MEDIUM       5.0        90.025                  90.910  0.885
            20          45     SOFT       7.0        91.722                  92.352  0.630
            22          45     SOFT       6.0        91.

In [58]:
print("===== LARGEST PREDICTION ERRORS =====")

print(
    comparison[
        [
            "driver_number",
            "lap_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "predicted_lap_duration",
            "error"
        ]
    ]
    .sort_values("error")
    .head(10)
    .round(3)
    .to_string(index=False)
)

print("\n===== MOST OVERPREDICTED LAPS =====")

print(
    comparison[
        [
            "driver_number",
            "lap_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "predicted_lap_duration",
            "error"
        ]
    ]
    .sort_values("error", ascending=False)
    .head(10)
    .round(3)
    .to_string(index=False)
)

===== LARGEST PREDICTION ERRORS =====
 driver_number  lap_number compound  tyre_age  lap_duration  predicted_lap_duration  error
            11          49     SOFT       1.0        97.360                  90.228 -7.132
             3          51     SOFT      13.0        94.814                  92.704 -2.110
             3          50     SOFT      12.0        94.150                  92.537 -1.613
            31          48   MEDIUM       9.0        95.931                  95.225 -0.706
            77          45     SOFT       7.0        93.159                  92.521 -0.638
             3          49     SOFT      11.0        92.926                  92.371 -0.555
            16          51     SOFT      13.0        91.674                  91.157 -0.517
             4          48     SOFT       8.0        90.556                  90.134 -0.422
            22          45     SOFT       6.0        91.438                  91.201 -0.237
             4          46     SOFT       6.0       

In [59]:
problem_drivers = [11, 31, 23, 81]

problem_context = pace_model_data[
    pace_model_data["driver_number"].isin(problem_drivers)
].copy()

print(
    problem_context[
        [
            "driver_number",
            "lap_number",
            "stint_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "fuel_effect",
            "tyre_degradation_effect",
            "traffic_ahead",
            "traffic_behind",
            "position_change",
            "position"
        ]
    ]
    .sort_values(["driver_number", "lap_number"])
    .tail(80)
    .round(3)
    .to_string(index=False)
)

 driver_number  lap_number  stint_number compound  tyre_age  lap_duration  fuel_effect  tyre_degradation_effect  traffic_ahead  traffic_behind  position_change  position
            23          44           3.0   MEDIUM       5.0        90.740       -3.617                    1.076            1.0             1.0              0.0      10.0
            23          45           3.0   MEDIUM       6.0        90.406       -3.701                    1.291            1.0             1.0              0.0      10.0
            23          46           3.0   MEDIUM       7.0        90.234       -3.785                    1.507            1.0             0.0             -1.0       9.0
            23          47           3.0   MEDIUM       8.0        90.148       -3.870                    1.722            1.0             0.0              0.0       9.0
            23          48           3.0   MEDIUM       9.0        90.068       -3.954                    1.937            1.0             0.0        

In [60]:
race_pace_by_lap = (
    pace_model_data
    .groupby("lap_number")["lap_duration"]
    .agg(
        median_lap_time="median",
        mean_lap_time="mean",
        driver_count="count"
    ).reset_index()
)
print(race_pace_by_lap.round(3).to_string(index=False))

 lap_number  median_lap_time  mean_lap_time  driver_count
          1          100.758        100.492            18
          2           92.942         92.977            19
          3           92.605         92.719            19
          4           92.968         92.748            19
          5           92.697         92.556            19
          6           92.697         92.669            19
          7           92.733         92.889            19
          8           92.798         92.859            19
          9           92.570         92.676            19
         10           92.647         92.743            19
         11           92.491         92.593            19
         12           92.427         92.513            19
         13           92.628         92.579            18
         14           92.488         92.460            19
         15           92.543         92.469            19
         16           93.178         93.266            19
         17   

In [61]:
race_pace_by_lap["rolling_median_3"] = (
    race_pace_by_lap["median_lap_time"]
    .rolling(window=3, min_periods=1)
    .median()
)

print(
    race_pace_by_lap[
        [
            "lap_number",
            "median_lap_time",
            "rolling_median_3",
            "driver_count"
        ]
    ].round(3).to_string(index=False)
)

 lap_number  median_lap_time  rolling_median_3  driver_count
          1          100.758           100.758            18
          2           92.942            96.850            19
          3           92.605            92.942            19
          4           92.968            92.942            19
          5           92.697            92.697            19
          6           92.697            92.697            19
          7           92.733            92.697            19
          8           92.798            92.733            19
          9           92.570            92.733            19
         10           92.647            92.647            19
         11           92.491            92.570            19
         12           92.427            92.491            19
         13           92.628            92.491            18
         14           92.488            92.488            19
         15           92.543            92.543            19
         16           93

In [62]:
race_pace_data = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
    & (enhanced_lap_data["is_pit_out_lap"] == False)
    & (enhanced_lap_data["compound"].isin(["SOFT","MEDIUM","HARD"]))
].copy()

race_pace_by_lap = (
    race_pace_data
    .groupby("lap_number")["lap_duration"]
    .median()
    .reset_index(name="median_lap_time")
)

full_laps = pd.DataFrame({
    "lap_number": range(
        int(race_pace_by_lap["lap_number"].min()),
        int(race_pace_by_lap["lap_number"].max())+1
    )
})
race_pace_by_lap = (
    full_laps
    .merge(race_pace_by_lap,on="lap_number",how="left")
    .sort_values("lap_number")
)

race_pace_by_lap["rolling_median_3"] = (
    race_pace_by_lap["median_lap_time"]
    .rolling(window=3, min_periods=1)
    .median()
)

print(
    race_pace_by_lap[
        ["lap_number", "median_lap_time", "rolling_median_3"]
    ]
    .round(3)
    .to_string(index=False)
)


 lap_number  median_lap_time  rolling_median_3
          1          100.758           100.758
          2           92.942            96.850
          3           92.605            92.942
          4           92.968            92.942
          5           92.697            92.697
          6           92.697            92.697
          7           92.733            92.697
          8           92.798            92.733
          9           92.570            92.733
         10           92.647            92.647
         11           92.491            92.570
         12           92.427            92.491
         13           92.628            92.491
         14           92.488            92.488
         15           92.543            92.543
         16           93.178            92.543
         17           95.014            93.178
         18           95.789            95.014
         19          104.151            95.789
         20           97.330            97.330
         21  

In [63]:
reference_pace = race_pace_by_lap.loc[
    race_pace_by_lap["lap_number"].between(5, 15),
    "median_lap_time"
].median()

print("Reference field pace:", round(reference_pace, 3), "seconds")

# Track evolution relative to that reference
race_pace_by_lap["track_evolution"] = (
    race_pace_by_lap["rolling_median_3"] - reference_pace
)

print(
    race_pace_by_lap[
        [
            "lap_number",
            "rolling_median_3",
            "track_evolution"
        ]
    ]
    .loc[
        race_pace_by_lap["lap_number"].between(39, 52)
    ]
    .round(3)
    .to_string(index=False)
)

Reference field pace: 92.628 seconds
 lap_number  rolling_median_3  track_evolution
         39            92.930            0.302
         40            92.321           -0.308
         41            91.712           -0.917
         42            90.649           -1.979
         43            90.164           -2.465
         44            90.164           -2.465
         45            90.211           -2.418
         46            90.211           -2.418
         47            90.218           -2.410
         48            90.218           -2.410
         49            90.365           -2.264
         50            90.516           -2.113
         51            90.732           -1.896
         52            90.516           -2.113


In [64]:
race_pace_data = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
    & (enhanced_lap_data["is_pit_out_lap"] == False)
    & enhanced_lap_data["compound"].isin(["SOFT","MEDIUM","HARD"])
].copy()

race_pace_by_lap = (
    race_pace_data
    .groupby("lap_number")["lap_duration"]
    .median()
    .reset_index(name="median_lap_time")
)

full_laps = pd.DataFrame({
    "lap_number": range(
        int(race_pace_by_lap["lap_number"].min()),
        int(race_pace_by_lap["lap_number"].max()) +1
    )
})

race_pace_by_lap = (
    full_laps
    .merge(
        race_pace_by_lap,
        on="lap_number",
        how="left"
    )
    .sort_values("lap_number")
    .reset_index(drop=True)
)

race_pace_by_lap["previous_lap_median"] = (
    race_pace_by_lap["median_lap_time"].shift(1)
)

race_pace_by_lap["recent_field_pace"] = (
    race_pace_by_lap["previous_lap_median"]
    .rolling(
        window=3,
        min_periods=1
    ).median()
)

reference_pace = race_pace_by_lap.loc[
    race_pace_by_lap["lap_number"].between(5,15),
    "median_lap_time"
].median()

print(
    "Reference field pace:",
    round(reference_pace,3),
    "seconds"
)

race_pace_by_lap["track_evolution"] = (
    race_pace_by_lap["recent_field_pace"] - reference_pace
)

print(
    race_pace_by_lap[
        [
            "lap_number",
            "median_lap_time",
            "recent_field_pace",
            "track_evolution"
        ]
    ]
    .loc[
        race_pace_by_lap["lap_number"].between(35,52)
    ]
    .round(3)
    .to_string(index=False)
)

Reference field pace: 92.628 seconds
 lap_number  median_lap_time  recent_field_pace  track_evolution
         35              NaN                NaN              NaN
         36              NaN                NaN              NaN
         37              NaN                NaN              NaN
         38              NaN                NaN              NaN
         39           92.930                NaN              NaN
         40           91.712             92.930            0.302
         41           90.649             92.321           -0.308
         42           90.164             91.712           -0.917
         43           90.144             90.649           -1.979
         44           90.211             90.164           -2.465
         45           90.227             90.164           -2.465
         46           90.160             90.211           -2.418
         47           90.218             90.211           -2.418
         48           90.365             90.218      

In [65]:
track_evolution_features = race_pace_by_lap[
    [
        "lap_number","track_evolution"
    ]
].copy()

enhanced_lap_data = enhanced_lap_data.merge(
    track_evolution_features,
    on="lap_number",
    how="left",
    validate="many_to_one"
)
print("Enhanced dataset shape:", enhanced_lap_data.shape)

print("\nNew columns:")
print(enhanced_lap_data.columns.tolist())

print("\nTrack evolution missing values:")
print(enhanced_lap_data["track_evolution"].isna().sum())

print("\nTrack evolution summary:")
print(
    enhanced_lap_data["track_evolution"]
    .describe()
)


Enhanced dataset shape: (962, 26)

New columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution']

Track evolution missing values:
167

Track evolution summary:
count    795.000000
mean       1.813453
std        5.238170
min       -2.464500
25%       -0.307500
50%        0.104500
75%        2.385500
max       20.347500
Name: track_evolution, dtype: float64


In [66]:
race_pace_reliability = (
    race_pace_data
    .groupby("lap_number")["lap_duration"]
    .agg(
        median_lap_time = "median",
        driver_count="count"
    ).reset_index()
)

race_pace_reliability = (
    race_pace_reliability
    .merge(
        race_pace_by_lap[
            [
                "lap_number",
                "recent_field_pace",
                "track_evolution"
            ]
        ],
        on="lap_number",
        how="left"
    )
    .sort_values("lap_number")
)

print(
    race_pace_reliability[
        [
            "lap_number",
            "driver_count",
            "median_lap_time",
            "recent_field_pace",
            "track_evolution"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

 lap_number  driver_count  median_lap_time  recent_field_pace  track_evolution
          1            18          100.758                NaN              NaN
          2            19           92.942            100.758            8.130
          3            19           92.605             96.850            4.221
          4            19           92.968             92.942            0.313
          5            19           92.697             92.942            0.313
          6            19           92.697             92.697            0.069
          7            19           92.733             92.697            0.069
          8            19           92.798             92.697            0.069
          9            19           92.570             92.733            0.105
         10            19           92.647             92.733            0.105
         11            19           92.491             92.647            0.019
         12            19           92.427          

In [67]:
# ============================================
# FILTER UNRELIABLE FIELD-PACE ESTIMATES
# ============================================

MIN_DRIVERS = 5

race_pace_reliability["reliable_median_lap_time"] = (
    race_pace_reliability["median_lap_time"]
    .where(
        race_pace_reliability["driver_count"] >= MIN_DRIVERS
    )
)


# Recalculate recent field pace using only
# reliable race-lap observations
race_pace_reliability["previous_reliable_pace"] = (
    race_pace_reliability["reliable_median_lap_time"].shift(1)
)


race_pace_reliability["recent_field_pace"] = (
    race_pace_reliability["previous_reliable_pace"]
    .rolling(
        window=3,
        min_periods=1
    )
    .median()
)


# --------------------------------------------
# Reference pace
# --------------------------------------------

reference_pace = race_pace_reliability.loc[
    race_pace_reliability["lap_number"].between(5, 15),
    "reliable_median_lap_time"
].median()

print(
    "Reference field pace:",
    round(reference_pace, 3),
    "seconds"
)


# --------------------------------------------
# Track evolution
# --------------------------------------------

race_pace_reliability["track_evolution"] = (
    race_pace_reliability["recent_field_pace"]
    - reference_pace
)


# --------------------------------------------
# Inspect suspicious late-race / transition laps
# --------------------------------------------

print(
    race_pace_reliability[
        [
            "lap_number",
            "driver_count",
            "reliable_median_lap_time",
            "recent_field_pace",
            "track_evolution"
        ]
    ]
    .loc[
        race_pace_reliability["lap_number"].between(20, 52)
    ]
    .round(3)
    .to_string(index=False)
)

Reference field pace: 92.628 seconds
 lap_number  driver_count  reliable_median_lap_time  recent_field_pace  track_evolution
         20            15                    97.330             95.789            3.160
         21            15                    93.484             97.330            4.701
         22            15                    92.495             97.330            4.701
         23            16                    92.752             93.484            0.855
         24            16                    93.792             92.752            0.123
         25            16                    95.542             92.752            0.123
         26            16                   102.870             93.792            1.164
         27             9                   108.126             95.542            2.914
         28             1                       NaN            102.870           10.242
         39             6                    92.930            105.498           12

In [68]:
# ============================================
# FINAL LEAKAGE-SAFE TRACK EVOLUTION
# ============================================

# Start from the reliable field pace
track_evolution_data = race_pace_reliability[
    [
        "lap_number",
        "reliable_median_lap_time"
    ]
].copy()


# --------------------------------------------
# Previous-lap information only
# --------------------------------------------

track_evolution_data["previous_pace"] = (
    track_evolution_data["reliable_median_lap_time"]
    .shift(1)
)


# --------------------------------------------
# Require 3 consecutive previous race laps
# --------------------------------------------
#
# Because the race timeline contains every lap,
# NaNs represent missing/unusable race laps.
#
# min_periods=3 prevents pandas from skipping
# those missing laps and reaching far backwards.

track_evolution_data["recent_field_pace"] = (
    track_evolution_data["previous_pace"]
    .rolling(
        window=3,
        min_periods=3
    )
    .median()
)


# --------------------------------------------
# Reference early-race pace
# --------------------------------------------

reference_pace = track_evolution_data.loc[
    track_evolution_data["lap_number"].between(5, 15),
    "reliable_median_lap_time"
].median()

print(
    "Reference field pace:",
    round(reference_pace, 3),
    "seconds"
)


# --------------------------------------------
# Track evolution
# --------------------------------------------

track_evolution_data["track_evolution"] = (
    track_evolution_data["recent_field_pace"]
    - reference_pace
)


# --------------------------------------------
# Inspect transition around lap 39+
# --------------------------------------------

print(
    track_evolution_data[
        [
            "lap_number",
            "reliable_median_lap_time",
            "previous_pace",
            "recent_field_pace",
            "track_evolution"
        ]
    ]
    .loc[
        track_evolution_data["lap_number"].between(35, 52)
    ]
    .round(3)
    .to_string(index=False)
)

Reference field pace: 92.628 seconds
 lap_number  reliable_median_lap_time  previous_pace  recent_field_pace  track_evolution
         39                    92.930            NaN                NaN              NaN
         40                    91.712         92.930                NaN              NaN
         41                    90.649         91.712                NaN              NaN
         42                    90.164         90.649             91.712           -0.917
         43                    90.144         90.164             90.649           -1.979
         44                    90.211         90.144             90.164           -2.465
         45                    90.227         90.211             90.164           -2.465
         46                    90.160         90.227             90.211           -2.418
         47                    90.218         90.160             90.211           -2.418
         48                    90.365         90.218             90.218  

In [69]:
enhanced_lap_data = enhanced_lap_data.drop(
    columns=["track_evolution"]
)
track_evolution_features = track_evolution_data[
    [
        "lap_number","track_evolution"
    ]
].copy()

enhanced_lap_data = enhanced_lap_data.merge(
    track_evolution_features,
    on="lap_number",
    how="left",
    validate="many_to_one"
)

print("Enhanced dataset shape:", enhanced_lap_data.shape)
print("\nTrack evolution missing values:")
print(
    enhanced_lap_data["track_evolution"].isna().sum()
)

print("\nTrack evolution summary:")
print(
    enhanced_lap_data["track_evolution"]
    .describe()
)

Enhanced dataset shape: (962, 26)

Track evolution missing values:
298

Track evolution summary:
count    664.000000
mean       0.279445
std        2.522388
min       -2.464500
25%       -1.896000
50%        0.068500
75%        0.549500
max       10.242000
Name: track_evolution, dtype: float64


In [70]:
pace_model_data = enhanced_lap_data[
    enhanced_lap_data["lap_duration"].notna()
    & (enhanced_lap_data["is_pit_out_lap"] == False)
    & enhanced_lap_data["compound"].isin(
        ["SOFT", "MEDIUM", "HARD"]
    )
    & (~enhanced_lap_data["lap_number"].isin([19, 25]))
].copy()


print("Pace model data shape:", pace_model_data.shape)

print("\nMissing values:")
print(
    pace_model_data[
        [
            "fuel_effect",
            "traffic_ahead",
            "traffic_behind",
            "position_change",
            "tyre_degradation_effect",
            "track_evolution"
        ]
    ]
    .isna()
    .sum()
)

Pace model data shape: (668, 26)

Missing values:
fuel_effect                11
traffic_ahead               0
traffic_behind              0
position_change             0
tyre_degradation_effect    18
track_evolution            94
dtype: int64


In [71]:
pace_model_data["track_evolution_available"] = (
    pace_model_data["track_evolution"].notna().astype(int)
)

pace_model_data["track_evolution"] = (
    pace_model_data["track_evolution"].fillna(0.0)
)

print("Missing track evolution:")

print(
    pace_model_data["track_evolution"].isna().sum()
)

print("Availability counts:")
print(
    pace_model_data["track_evolution_available"]
    .value_counts()
    .sort_index()
)

print("Track evolution summary:")
print(
    pace_model_data["track_evolution"]
    .describe()
)

Missing track evolution:
0
Availability counts:
track_evolution_available
0     94
1    574
Name: count, dtype: int64
Track evolution summary:
count    668.000000
mean      -0.191059
std        1.701435
min       -2.464500
25%       -1.896000
50%        0.000000
75%        0.104500
max       10.242000
Name: track_evolution, dtype: float64


In [72]:
print(
    pace_model_data[
        [
            "lap_number",
            "track_evolution",
            "track_evolution_available"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "track_evolution",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)

 lap_number  track_evolution  track_evolution_available
         28          10.2420                          1
         22           4.7015                          1
         21           4.7015                          1
         20           3.1605                          1
         27           2.9140                          1
         26           1.1640                          1
         23           0.8555                          1
         18           0.5495                          1
          5           0.3135                          1
          4           0.3135                          1
         24           0.1230                          1
          9           0.1045                          1
         10           0.1045                          1
          6           0.0685                          1
          7           0.0685                          1


In [73]:
train_data = pace_model_data[
    pace_model_data["lap_number"] < 45
].copy()

test_data = pace_model_data[
    pace_model_data["lap_number"] >= 45
].copy()


print("Training rows:", len(train_data))
print("Test rows:", len(test_data))

model_without_track = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    """,
    data=train_data
).fit()

pred_without_track = model_without_track.predict(
    test_data
)

model_with_track = smf.ols(
    """
    lap_duration ~
    C(driver_number)
    + C(compound)
    + fuel_effect
    + traffic_ahead
    + traffic_behind
    + position_change
    + tyre_degradation_effect
    + track_evolution
    + track_evolution_available
    """,
    data=train_data
).fit()


pred_with_track = model_with_track.predict(
    test_data
)

rmse_without_track = np.sqrt(
    np.mean(
        (test_data["lap_duration"] - pred_without_track) ** 2
    )
)

rmse_with_track = np.sqrt(
    np.mean(
        (test_data["lap_duration"] - pred_with_track) ** 2
    )
)
r2_without_track = (
    1
    - (
        np.sum(
            (test_data["lap_duration"] - pred_without_track) ** 2
        )
        /
        np.sum(
            (test_data["lap_duration"] - test_data["lap_duration"].mean()) ** 2
        )
    )
)
r2_with_track = (
    1
    - (
        np.sum(
            (test_data["lap_duration"] - pred_with_track) ** 2
        )
        /
        np.sum(
            (test_data["lap_duration"] - test_data["lap_duration"].mean()) ** 2
        )
    )
)
comparison_results = pd.DataFrame({
    "model": [
        "Without track evolution",
        "With track evolution"
    ],
    "RMSE": [
        rmse_without_track,
        rmse_with_track
    ],
    "R2": [
        r2_without_track,
        r2_with_track
    ]
})


print(
    comparison_results.round(4).to_string(index=False)
)

Training rows: 535
Test rows: 133
                  model   RMSE      R2
Without track evolution 1.4461 -0.2418
   With track evolution 1.3247 -0.0420


In [74]:
from pathlib import Path

project = Path.cwd()

print("Notebook is running from:")
print(project)

print("\nContents of current directory:")
for item in project.iterdir():
    print(item)

print("\nExpected weather path:")
weather_path = project / "data" / "processed" / "9558" / "weather.parquet"
print(weather_path)

print("\nDoes expected path exist?")
print(weather_path.exists())

print("\nDoes data/processed/9558 exist?")
print((project / "data" / "processed" / "9558").exists())

if (project / "data" / "processed" / "9558").exists():
    print("\nContents of data/processed/9558:")
    for item in (project / "data" / "processed" / "9558").iterdir():
        print(item.name)

Notebook is running from:
c:\Users\Nihan\.vscode\pitwall\notebooks

Contents of current directory:
c:\Users\Nihan\.vscode\pitwall\notebooks\01_data_exploration.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\02_race_analysis.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\03_traffic_features.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\04_fuel_effects.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\05_tyre_degradation.ipynb
c:\Users\Nihan\.vscode\pitwall\notebooks\06_enhanced_pace_model.ipynb

Expected weather path:
c:\Users\Nihan\.vscode\pitwall\notebooks\data\processed\9558\weather.parquet

Does expected path exist?
False

Does data/processed/9558 exist?
False


In [75]:
import pandas as pd

weather_path = "../data/processed/9558/weather.parquet"

weather = pd.read_parquet(weather_path)

print("Shape:", weather.shape)

print("\nColumns:")
print(weather.columns.tolist())

print("\nFirst 5 rows:")
display(weather.head())

print("\nDtypes:")
print(weather.dtypes)

print("\nMissing values:")
print(weather.isna().sum())

date_cols = [
    col for col in weather.columns
    if "date" in col.lower() or "time" in col.lower()
]

print("\nTimestamp/date columns:", date_cols)

for col in date_cols:
    dt = pd.to_datetime(weather[col])

    print(f"\n{col}:")
    print("  Minimum:", dt.min())
    print("  Maximum:", dt.max())
    print("  Unique values:", dt.nunique())

    if len(dt) > 1:
        print("\n  Most common time intervals:")
        print(dt.sort_values().diff().value_counts().head(10))

Shape: (147, 10)

Columns:
['date', 'session_key', 'meeting_key', 'track_temperature', 'wind_speed', 'rainfall', 'humidity', 'pressure', 'air_temperature', 'wind_direction']

First 5 rows:


,date,session_key,meeting_key,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction
0,2024-07-07 13:05:02.119000+00:00,9558,1240,28.1,2.2,0,67.0,991.8,15.9,226
1,2024-07-07 13:06:02.098000+00:00,9558,1240,27.6,1.5,0,66.0,991.7,15.9,268
2,2024-07-07 13:07:02.122000+00:00,9558,1240,27.0,1.4,0,65.0,991.7,15.9,291
3,2024-07-07 13:08:02.127000+00:00,9558,1240,27.0,1.1,0,66.0,991.8,15.9,195
4,2024-07-07 13:09:02.125000+00:00,9558,1240,31.0,1.9,0,64.0,991.8,15.9,225



Dtypes:
date                 datetime64[us, UTC]
session_key                        int64
meeting_key                        int64
track_temperature                float64
wind_speed                       float64
rainfall                           int64
humidity                         float64
pressure                         float64
air_temperature                  float64
wind_direction                     int64
dtype: object

Missing values:
date                 0
session_key          0
meeting_key          0
track_temperature    0
wind_speed           0
rainfall             0
humidity             0
pressure             0
air_temperature      0
wind_direction       0
dtype: int64

Timestamp/date columns: ['date']

date:
  Minimum: 2024-07-07 13:05:02.119000+00:00
  Maximum: 2024-07-07 15:31:03.204000+00:00
  Unique values: 147

  Most common time intervals:
date
0 days 00:01:00.020000    7
0 days 00:01:00.005000    6
0 days 00:01:00.014000    6
0 days 00:01:00           6
0 days 00

In [76]:
print("lap data enhanced shape:", enhanced_lap_data.shape)
print("Columns containing date/time")
print([
    col for col  in enhanced_lap_data.columns
    if "date" in col.lower() or "time" in col.lower()
])

print("First 5 lap records:")
display(
    enhanced_lap_data[
        [col for col in enhanced_lap_data.columns
        if "date" in col.lower() or "time" in col.lower()]
        +["lap_number"]
    ].head()
)

lap data enhanced shape: (962, 26)
Columns containing date/time
['date_start']
First 5 lap records:


,date_start,lap_number
0,2024-07-07 14:03:12.540000+00:00,1
1,2024-07-07 14:04:49.750000+00:00,2
2,2024-07-07 14:06:21.461000+00:00,3
3,2024-07-07 14:07:53.675000+00:00,4
4,2024-07-07 14:09:25.527000+00:00,5


In [115]:
print("Null date_start:", enhanced_lap_data["date_start"].isna().sum())
print("Null weather date:", weather["date"].isna().sum())

print("\nNull session_key in laps:", enhanced_lap_data["session_key"].isna().sum())
print("Null session_key in weather:", weather["session_key"].isna().sum())

print(
    enhanced_lap_data[
        enhanced_lap_data["date_start"].isna()
    ][["session_key", "lap_number", "date_start"]].head(20)
)

Null date_start: 1
Null weather date: 0

Null session_key in laps: 0
Null session_key in weather: 0
     session_key  lap_number date_start
208         9558           1        NaT


In [116]:

weather_check = weather[
    ["date", "session_key"]
].sort_values(["session_key", "date"]).copy()

lap_check = enhanced_lap_data[
    ["date_start", "session_key", "lap_number"]
].dropna(subset=["date_start"]).sort_values(
    ["session_key", "date_start"]
).copy()

aligned_check = pd.merge_asof(
    lap_check,
    weather_check,
    left_on="date_start",
    right_on="date",
    by="session_key",
    direction="backward"
)

aligned_check["weather_lag_seconds"] = (
    aligned_check["date_start"] - aligned_check["date"]
).dt.total_seconds()

print("Weather alignment lag:")
print(aligned_check["weather_lag_seconds"].describe())

print(
    "\nMaximum lag:",
    aligned_check["weather_lag_seconds"].max(),
    "seconds"
)

print("\nFirst 10 aligned laps:")

display(
    aligned_check[
        [
            "session_key",
            "lap_number",
            "date_start",
            "date",
            "weather_lag_seconds"
        ]
    ].head(10)
)

Weather alignment lag:
count    961.000000
mean      29.377612
std       17.568594
min        0.019000
25%       13.337000
50%       29.296000
75%       44.449000
max       59.981000
Name: weather_lag_seconds, dtype: float64

Maximum lag: 59.981 seconds

First 10 aligned laps:


,session_key,lap_number,date_start,date,weather_lag_seconds
0,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044
1,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044
2,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044
3,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044
4,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044
5,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044
6,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044
7,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044
8,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044
9,9558,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044


In [117]:

print("Total laps:", len(enhanced_lap_data))

print("\nNull date_start values:")
print(enhanced_lap_data["date_start"].isna().sum())

print("\nRows with null date_start:")
display(
    enhanced_lap_data[
        enhanced_lap_data["date_start"].isna()
    ]
)

print("\nNull session_key values:")
print(enhanced_lap_data["session_key"].isna().sum())

Total laps: 962

Null date_start values:
1

Rows with null date_start:


,meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed,stint_number,compound,tyre_age,fuel_effect,position,traffic_ahead,traffic_behind,position_change,tyre_degradation_effect,track_evolution
208,1240,9558,11,1,NaT,NaN,39.507,26.165,288.0,252.0,False,NaN,"[nan, 2064.0, 2049.0, 2049.0, 2049.0, 2049.0, ...","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 2049]",293.0,1.0,HARD,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Null session_key values:
0


In [118]:
lap_weather = enhanced_lap_data[
    ["meeting_key", "session_key","driver_number","lap_number","date_start"]
].copy(0)

lap_weather_valid = lap_weather.dropna(subset=["date_start"]).copy()

lap_weather_valid = lap_weather.dropna(subset=["date_start"]).copy()

weather_alignment = weather[
    ["session_key", "date",
     "track_temperature",
     "wind_speed",
     "rainfall",
     "humidity",
     "pressure",
     "air_temperature",
     "wind_direction"]
].copy()

lap_weather_valid["date_start"] = pd.to_datetime(
    lap_weather_valid["date_start"], utc=True
)

weather_alignment["date"] = pd.to_datetime(
    weather_alignment["date"], utc=True
)

lap_weather_valid = lap_weather_valid.sort_values(
    ["session_key", "date_start"]
)

weather_alignment = weather_alignment.sort_values(
    ["session_key", "date"]
)

print("Valid lap rows:", len(lap_weather_valid))
print("Weather rows:", len(weather_alignment))

print("\nLap sessions:")
print(lap_weather_valid["session_key"].unique())

print("\nWeather sessions:")
print(weather_alignment["session_key"].unique())

Valid lap rows: 961
Weather rows: 147

Lap sessions:
[9558]

Weather sessions:
[9558]


In [119]:
lap_weather_aligned = pd.merge_asof(
   lap_weather_valid,
   weather_alignment,
   left_on="date_start",
   right_on="date",
   by="session_key",
   direction="backward" 
)

lap_weather_aligned["weather_lag_seconds"] = (
    lap_weather_aligned["date_start"]
    - lap_weather_aligned["date"]
).dt.total_seconds()

print("Aligned shape:", lap_weather_aligned.shape)

print("\nWeather lag statistics:")
print(lap_weather_aligned["weather_lag_seconds"].describe())

print("\nMaximum weather lag:",
      lap_weather_aligned["weather_lag_seconds"].max(),
      "seconds")

print("\nFirst 10 aligned laps:")
display(
    lap_weather_aligned[
        ["lap_number", "date_start", "date",
         "weather_lag_seconds",
         "track_temperature",
         "air_temperature",
         "rainfall"]
    ].head(10)
)

Aligned shape: (961, 14)

Weather lag statistics:
count    961.000000
mean      29.377612
std       17.568594
min        0.019000
25%       13.337000
50%       29.296000
75%       44.449000
max       59.981000
Name: weather_lag_seconds, dtype: float64

Maximum weather lag: 59.981 seconds

First 10 aligned laps:


,lap_number,date_start,date,weather_lag_seconds,track_temperature,air_temperature,rainfall
0,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
1,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
2,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
3,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
4,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
5,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
6,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
7,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
8,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0
9,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,33.0,16.1,0


In [120]:
weather_features = [
    "track_temperature",
    "wind_speed",
    "rainfall",
    "humidity",
    "pressure",
    "air_temperature",
    "wind_direction"
]

print("Unique values and ranges:")

for col in weather_features:
    print(f"{col}:")
    print(f"  Unique values: {weather[col].nunique()}")
    print(f"  Min: {weather[col].min()}")
    print(f"  Max: {weather[col].max()}")
    
    if weather[col].nunique() <= 15:
        print(f"  Values: {sorted(weather[col].unique())}")
    
    print()

Unique values and ranges:
track_temperature:
  Unique values: 68
  Min: 20.7
  Max: 37.9

wind_speed:
  Unique values: 21
  Min: 0.4
  Max: 2.5

rainfall:
  Unique values: 2
  Min: 0
  Max: 1
  Values: [np.int64(0), np.int64(1)]

humidity:
  Unique values: 20
  Min: 59.0
  Max: 78.0

pressure:
  Unique values: 10
  Min: 991.7
  Max: 992.6
  Values: [np.float64(991.7), np.float64(991.8), np.float64(991.9), np.float64(992.0), np.float64(992.1), np.float64(992.2), np.float64(992.3), np.float64(992.4), np.float64(992.5), np.float64(992.6)]

air_temperature:
  Unique values: 24
  Min: 14.5
  Max: 16.8

wind_direction:
  Unique values: 87
  Min: 0
  Max: 335



In [121]:
print("Rainfall observations:")

display(
    weather.loc[
        weather["rainfall"] == 1,
        ["date", "track_temperature", "air_temperature", "humidity", "wind_speed", "rainfall"]
    ]
)
print("\nRainfall count:")
print(weather["rainfall"].value_counts())

print("\nWeather observations over time:")
display(
    weather[
        ["date", "track_temperature", "wind_speed",
         "rainfall", "humidity", "pressure",
         "air_temperature", "wind_direction"]
    ].head(20)
)

Rainfall observations:


,date,track_temperature,air_temperature,humidity,wind_speed,rainfall
5,2024-07-07 13:10:02.120000+00:00,31.3,15.9,64.0,2.0,1
6,2024-07-07 13:11:02.102000+00:00,31.3,15.8,64.0,2.0,1
7,2024-07-07 13:12:02.096000+00:00,31.0,15.8,67.0,1.1,1
8,2024-07-07 13:13:02.111000+00:00,26.5,15.8,68.0,1.0,1
9,2024-07-07 13:14:02.125000+00:00,26.5,15.8,70.0,0.6,1
10,2024-07-07 13:15:02.145000+00:00,26.6,15.7,70.0,0.7,1
11,2024-07-07 13:16:02.153000+00:00,26.5,15.6,70.0,0.7,1
12,2024-07-07 13:17:02.146000+00:00,26.5,15.5,71.0,0.7,1
13,2024-07-07 13:18:02.160000+00:00,27.6,15.4,75.0,1.0,1
14,2024-07-07 13:19:02.145000+00:00,29.4,15.4,73.0,0.6,1



Rainfall count:
rainfall
0    96
1    51
Name: count, dtype: int64

Weather observations over time:


,date,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction
0,2024-07-07 13:05:02.119000+00:00,28.1,2.2,0,67.0,991.8,15.9,226
1,2024-07-07 13:06:02.098000+00:00,27.6,1.5,0,66.0,991.7,15.9,268
2,2024-07-07 13:07:02.122000+00:00,27.0,1.4,0,65.0,991.7,15.9,291
3,2024-07-07 13:08:02.127000+00:00,27.0,1.1,0,66.0,991.8,15.9,195
4,2024-07-07 13:09:02.125000+00:00,31.0,1.9,0,64.0,991.8,15.9,225
5,2024-07-07 13:10:02.120000+00:00,31.3,2.0,1,64.0,991.9,15.9,206
6,2024-07-07 13:11:02.102000+00:00,31.3,2.0,1,64.0,991.9,15.8,206
7,2024-07-07 13:12:02.096000+00:00,31.0,1.1,1,67.0,991.8,15.8,242
8,2024-07-07 13:13:02.111000+00:00,26.5,1.0,1,68.0,991.8,15.8,254
9,2024-07-07 13:14:02.125000+00:00,26.5,0.6,1,70.0,991.8,15.8,0


In [122]:
print("Weather time range:")
print("Start:", weather["date"].min())
print("End:",weather["date"].max())

print("\nWeather observations around race start:")

race_start = enhanced_lap_data["date_start"].min()

display(
    weather[
        (weather["date"] >= race_start - pd.Timedelta(minutes=5)) &
        (weather["date"] <= race_start + pd.Timedelta(minutes=5))
    ]
)

print("Weather observations around race end:")

race_end = enhanced_lap_data["date_start"].max()

display(
    weather[
        (weather["date"] >= race_end - pd.Timedelta(minutes=5)) &
        (weather["date"] <= race_end + pd.Timedelta(minutes=5))
    ]
)

Weather time range:
Start: 2024-07-07 13:05:02.119000+00:00
End: 2024-07-07 15:31:03.204000+00:00

Weather observations around race start:


,date,session_key,meeting_key,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction
54,2024-07-07 13:59:02.452000+00:00,9558,1240,33.0,1.7,0,64.0,992.0,16.1,243
55,2024-07-07 14:00:02.446000+00:00,9558,1240,33.0,1.8,0,63.0,992.0,16.1,238
56,2024-07-07 14:01:02.472000+00:00,9558,1240,33.0,2.0,0,64.0,992.0,16.1,233
57,2024-07-07 14:02:02.478000+00:00,9558,1240,33.0,1.9,0,63.0,992.0,16.1,254
58,2024-07-07 14:03:02.496000+00:00,9558,1240,33.0,1.6,0,64.0,992.0,16.1,275
59,2024-07-07 14:04:02.504000+00:00,9558,1240,33.0,1.6,0,63.0,992.0,16.2,215
60,2024-07-07 14:05:02.495000+00:00,9558,1240,33.0,1.8,0,61.0,991.9,16.2,249
61,2024-07-07 14:06:02.504000+00:00,9558,1240,33.0,2.0,0,62.0,991.9,16.3,235
62,2024-07-07 14:07:02.521000+00:00,9558,1240,24.3,1.9,0,61.0,992.0,16.3,237
63,2024-07-07 14:08:02.584000+00:00,9558,1240,24.1,2.5,0,61.0,991.9,16.3,216


Weather observations around race end:


,date,session_key,meeting_key,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction
136,2024-07-07 15:21:03.104000+00:00,9558,1240,25.2,1.8,0,69.0,992.4,15.8,222
137,2024-07-07 15:22:03.113000+00:00,9558,1240,25.4,1.7,0,69.0,992.5,15.8,219
138,2024-07-07 15:23:03.118000+00:00,9558,1240,25.5,1.4,0,69.0,992.5,15.9,230
139,2024-07-07 15:24:03.133000+00:00,9558,1240,25.3,1.2,0,68.0,992.4,15.9,258
140,2024-07-07 15:25:03.120000+00:00,9558,1240,25.2,0.7,0,69.0,992.5,15.8,241
141,2024-07-07 15:26:03.163000+00:00,9558,1240,24.9,1.9,0,68.0,992.5,15.8,230
142,2024-07-07 15:27:03.155000+00:00,9558,1240,24.8,2.1,0,69.0,992.4,15.8,224
143,2024-07-07 15:28:03.175000+00:00,9558,1240,25.0,1.7,0,70.0,992.4,15.9,261
144,2024-07-07 15:29:03.195000+00:00,9558,1240,26.1,1.0,0,71.0,992.4,15.9,222
145,2024-07-07 15:30:03.198000+00:00,9558,1240,26.4,1.6,0,69.0,992.5,16.0,230


In [123]:
# Inspect unusually large minute-to-minute weather changes

weather_sorted = weather.sort_values("date").copy()

numeric_weather = [
    "track_temperature",
    "wind_speed",
    "humidity",
    "pressure",
    "air_temperature",
    "wind_direction"
]

for col in numeric_weather:
    weather_sorted[f"{col}_change"] = weather_sorted[col].diff()

print("Largest absolute changes between consecutive weather observations:\n")

for col in numeric_weather:
    change_col = f"{col}_change"

    idx = weather_sorted[change_col].abs().idxmax()
    row = weather_sorted.loc[idx]

    print(f"{col}:")
    print(f"  Change: {row[change_col]}")
    print(f"  Time:   {row['date']}")
    print()

print("Rainfall transitions:")

rain_change = weather_sorted["rainfall"].diff()
display(
    weather_sorted.loc[
        rain_change.fillna(0) != 0,
        ["date", "rainfall", "track_temperature",
         "air_temperature", "humidity", "wind_speed"]
    ]
)

Largest absolute changes between consecutive weather observations:

track_temperature:
  Change: -8.7
  Time:   2024-07-07 14:07:02.521000+00:00

wind_speed:
  Change: 1.6
  Time:   2024-07-07 13:35:02.291000+00:00

humidity:
  Change: -5.0
  Time:   2024-07-07 13:29:02.220000+00:00

pressure:
  Change: 0.20000000000004547
  Time:   2024-07-07 13:56:02.427000+00:00

air_temperature:
  Change: -0.3000000000000007
  Time:   2024-07-07 14:35:02.692000+00:00

wind_direction:
  Change: 327.0
  Time:   2024-07-07 13:16:02.153000+00:00

Rainfall transitions:


,date,rainfall,track_temperature,air_temperature,humidity,wind_speed
5,2024-07-07 13:10:02.120000+00:00,1,31.3,15.9,64.0,2.0
17,2024-07-07 13:22:02.173000+00:00,0,32.9,15.3,72.0,1.0
21,2024-07-07 13:26:02.189000+00:00,1,34.5,15.8,71.0,0.8
23,2024-07-07 13:28:02.230000+00:00,0,36.9,16.0,71.0,1.0
85,2024-07-07 14:30:02.680000+00:00,1,23.9,16.6,60.0,1.6
120,2024-07-07 15:05:02.943000+00:00,0,24.6,14.6,78.0,1.4
128,2024-07-07 15:13:03.028000+00:00,1,24.5,15.4,72.0,2.0
130,2024-07-07 15:15:03.048000+00:00,0,25.0,15.5,73.0,2.0


In [124]:
print("Lap level weather summary:")

weather_features = [
    "track_temperature",
    "wind_speed",
    "rainfall",
    "humidity",
    "pressure",
    "air_temperature",
    "wind_direction",
    "weather_lag_seconds"
]

display(
    lap_weather_aligned[weather_features].describe()
)
print("\nRainfall at lap start:")
display(
    lap_weather_aligned["rainfall"].value_counts(dropna=False)
)

print("Laps aligned to rainfall:")

display(
    lap_weather_aligned.loc[
        lap_weather_aligned["rainfall"] == 1,
        [
            "driver_number",
            "lap_number",
            "date_start",
            "date",
            "rainfall",
            "track_temperature",
            "air_temperature",
            "humidity",
            "wind_speed",
            "weather_lag_seconds"
        ]
    ].drop_duplicates(
        subset=["lap_number", "date"]
    ).sort_values("date")
)

print("Track temperature around 14:07 anomaly:")
display(
    weather.loc[
        (weather["date"] >= "2024-07-07 14:04:00+00:00") &
        (weather["date"] <= "2024-07-07 14:10:00+00:00"),
        [
            "date",
            "track_temperature",
            "air_temperature",
            "humidity",
            "wind_speed",
            "rainfall"
        ]
    ]
)

Lap level weather summary:


,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction,weather_lag_seconds
count,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000,961.000000
mean,24.125494,1.483039,0.423517,67.522373,992.246410,15.651301,232.952133,29.377612
std,2.796364,0.470786,0.494373,5.822708,0.251806,0.728670,40.770442,17.568594
min,20.700000,0.500000,0.000000,59.000000,991.800000,14.500000,59.000000,0.019000
25%,22.000000,1.100000,0.000000,61.000000,992.000000,15.100000,219.000000,13.337000
50%,24.300000,1.500000,0.000000,68.000000,992.200000,15.600000,237.000000,29.296000
75%,25.200000,1.900000,1.000000,73.000000,992.500000,16.300000,251.000000,44.449000
max,33.000000,2.500000,1.000000,78.000000,992.600000,16.800000,335.000000,59.981000



Rainfall at lap start:


rainfall
0    554
1    407
Name: count, dtype: int64

Laps aligned to rainfall:


,driver_number,lap_number,date_start,date,rainfall,track_temperature,air_temperature,humidity,wind_speed,weather_lag_seconds
340,31,18,2024-07-07 14:30:08.240000+00:00,2024-07-07 14:30:02.680000+00:00,1,23.9,16.6,60.0,1.6,5.560
342,44,19,2024-07-07 14:30:52.867000+00:00,2024-07-07 14:30:02.680000+00:00,1,23.9,16.6,60.0,1.6,50.187
347,55,19,2024-07-07 14:31:03.691000+00:00,2024-07-07 14:31:02.667000+00:00,1,23.2,16.4,63.0,1.5,1.024
360,24,19,2024-07-07 14:32:06.641000+00:00,2024-07-07 14:32:02.688000+00:00,1,22.9,16.2,64.0,1.5,3.953
361,44,20,2024-07-07 14:32:36.572000+00:00,2024-07-07 14:32:02.688000+00:00,1,22.9,16.2,64.0,1.5,33.884
...,...,...,...,...,...,...,...,...,...,...
811,18,44,2024-07-07 15:13:06.495000+00:00,2024-07-07 15:13:03.028000+00:00,1,24.5,15.4,72.0,2.0,3.467
820,44,45,2024-07-07 15:13:42.570000+00:00,2024-07-07 15:13:03.028000+00:00,1,24.5,15.4,72.0,2.0,39.542
825,55,45,2024-07-07 15:14:09.436000+00:00,2024-07-07 15:14:03.040000+00:00,1,24.5,15.4,73.0,2.0,6.396
824,77,44,2024-07-07 15:14:06.347000+00:00,2024-07-07 15:14:03.040000+00:00,1,24.5,15.4,73.0,2.0,3.307


Track temperature around 14:07 anomaly:


,date,track_temperature,air_temperature,humidity,wind_speed,rainfall
59,2024-07-07 14:04:02.504000+00:00,33.0,16.2,63.0,1.6,0
60,2024-07-07 14:05:02.495000+00:00,33.0,16.2,61.0,1.8,0
61,2024-07-07 14:06:02.504000+00:00,33.0,16.3,62.0,2.0,0
62,2024-07-07 14:07:02.521000+00:00,24.3,16.3,61.0,1.9,0
63,2024-07-07 14:08:02.584000+00:00,24.1,16.3,61.0,2.5,0
64,2024-07-07 14:09:02.537000+00:00,24.0,16.2,60.0,2.2,0


In [125]:
# Inspect relationship between weather conditions and lap duration

weather_pace = lap_weather_aligned.merge(
    enhanced_lap_data[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "compound"
        ]
    ],
    on=["driver_number", "lap_number"],
    how="left"
)

print("Weather + pace shape:", weather_pace.shape)

print("\nLap duration by rainfall:")
display(
    weather_pace.groupby("rainfall")["lap_duration"].agg(
        ["count", "mean", "median", "std", "min", "max"]
    )
)

print("\nLap duration by rainfall and compound:")
display(
    weather_pace.groupby(["rainfall", "compound"])["lap_duration"].agg(
        ["count", "mean", "median", "std"]
    )
)

print("\nLap duration around track-temperature anomaly:")

display(
    weather_pace.loc[
        (weather_pace["date_start"] >= "2024-07-07 14:05:00+00:00") &
        (weather_pace["date_start"] <= "2024-07-09 14:10:00+00:00"),
        [
            "driver_number",
            "lap_number",
            "date_start",
            "lap_duration",
            "compound",
            "track_temperature",
            "rainfall"
        ]
    ].sort_values("date_start").head(40)
)

Weather + pace shape: (961, 16)

Lap duration by rainfall:


,count,mean,median,std,min,max
rainfall,,,,,,
0,553,93.088911,92.361,4.686977,88.293,126.729
1,406,101.933342,102.441,7.868429,89.137,128.459



Lap duration by rainfall and compound:


count        mean    median        std
rainfall compound                                            
0        HARD             39   91.766179   90.0560   4.509557
         INTERMEDIATE      1   99.440000   99.4400        NaN
         MEDIUM          359   93.208312   92.5900   3.933854
         SOFT            154   93.104305   91.3210   6.103452
1        HARD              4   99.726250   95.7285  13.721954
         INTERMEDIATE    233  105.011146  103.2290   6.497258
         MEDIUM          149   97.521443   95.2350   6.298948
         SOFT             20   99.387000   91.3095  13.740177


Lap duration around track-temperature anomaly:


,driver_number,lap_number,date_start,lap_duration,compound,track_temperature,rainfall
37,11,2,2024-07-07 14:05:02.418000+00:00,92.942,HARD,33.0,0
38,63,3,2024-07-07 14:06:19.683000+00:00,91.298,MEDIUM,33.0,0
39,44,3,2024-07-07 14:06:20.512000+00:00,91.716,MEDIUM,33.0,0
40,1,3,2024-07-07 14:06:21.461000+00:00,92.200,MEDIUM,33.0,0
41,4,3,2024-07-07 14:06:22.424000+00:00,91.991,MEDIUM,33.0,0
42,81,3,2024-07-07 14:06:23.337000+00:00,91.927,MEDIUM,33.0,0
43,55,3,2024-07-07 14:06:24.036000+00:00,92.102,MEDIUM,33.0,0
44,18,3,2024-07-07 14:06:24.689000+00:00,92.307,MEDIUM,33.0,0
45,16,3,2024-07-07 14:06:25.521000+00:00,92.369,MEDIUM,33.0,0
46,27,3,2024-07-07 14:06:25.948000+00:00,92.605,MEDIUM,33.0,0


In [126]:
# Check candidate weather features before adding them to the pace model

candidate_weather = [
    "rainfall",
    "humidity",
    "air_temperature",
    "wind_speed"
]

print("Candidate weather feature statistics:\n")

display(
    lap_weather_aligned[candidate_weather].describe()
)

print("\nUnique values:\n")

for col in candidate_weather:
    print(f"{col}: {lap_weather_aligned[col].nunique()} unique values")

print("\nCorrelation matrix:\n")

display(
    lap_weather_aligned[candidate_weather].corr()
)

print("\nWeather conditions by race lap:")

display(
    lap_weather_aligned.groupby("lap_number")[candidate_weather]
    .first()
    .head(25)
)

Candidate weather feature statistics:



,rainfall,humidity,air_temperature,wind_speed
count,961.000000,961.000000,961.000000,961.000000
mean,0.423517,67.522373,15.651301,1.483039
std,0.494373,5.822708,0.728670,0.470786
min,0.000000,59.000000,14.500000,0.500000
25%,0.000000,61.000000,15.100000,1.100000
50%,0.000000,68.000000,15.600000,1.500000
75%,1.000000,73.000000,16.300000,1.900000
max,1.000000,78.000000,16.800000,2.500000



Unique values:

rainfall: 2 unique values
humidity: 18 unique values
air_temperature: 23 unique values
wind_speed: 20 unique values

Correlation matrix:



,rainfall,humidity,air_temperature,wind_speed
rainfall,1.000000,0.464418,-0.664438,-0.425167
humidity,0.464418,1.000000,-0.904431,-0.379536
air_temperature,-0.664438,-0.904431,1.000000,0.462054
wind_speed,-0.425167,-0.379536,0.462054,1.000000



Weather conditions by race lap:


,rainfall,humidity,air_temperature,wind_speed
lap_number,,,,
1,0,64.0,16.1,1.6
2,0,63.0,16.2,1.6
3,0,62.0,16.3,2.0
4,0,61.0,16.3,1.9
5,0,60.0,16.2,2.2
6,0,60.0,16.3,2.4
7,0,61.0,16.3,2.1
8,0,60.0,16.4,2.4
9,0,61.0,16.5,2.0


In [127]:
weather_features = lap_weather_aligned[
    [
        "driver_number",
        "lap_number",
        "rainfall",
        "humidity",
        "air_temperature",
        "wind_speed",
        "weather_lag_seconds"
    ]
].copy()

print("Weather feature rows:", len(weather_features))
print("Unique driver-lap pairs:",
      weather_features[["driver_number", "lap_number"]].drop_duplicates().shape[0])

print("\nDuplicate driver-lap pairs:")
print(
    weather_features.duplicated(
        subset=["driver_number", "lap_number"]
    ).sum()
)

Weather feature rows: 961
Unique driver-lap pairs: 961

Duplicate driver-lap pairs:
0


In [128]:
enhanced_lap_data_weather = enhanced_lap_data.merge(
    weather_features,
    on=["driver_number", "lap_number"],
    how="left",
    validate="one_to_one"
)

print("Original shape:", enhanced_lap_data.shape)
print("New shape:", enhanced_lap_data_weather.shape)

print("\nWeather feature missing values:")
display(
    enhanced_lap_data_weather[
        [
            "rainfall",
            "humidity",
            "air_temperature",
            "wind_speed",
            "weather_lag_seconds"
        ]
    ].isna().sum()
)

print("\nWeather columns:")
display(
    enhanced_lap_data_weather[
        [
            "driver_number",
            "lap_number",
            "date_start",
            "rainfall",
            "humidity",
            "air_temperature",
            "wind_speed",
            "weather_lag_seconds"
        ]
    ].head(10)
)

Original shape: (962, 26)
New shape: (962, 31)

Weather feature missing values:


rainfall               1
humidity               1
air_temperature        1
wind_speed             1
weather_lag_seconds    1
dtype: int64


Weather columns:


,driver_number,lap_number,date_start,rainfall,humidity,air_temperature,wind_speed,weather_lag_seconds
0,1,1,2024-07-07 14:03:12.540000+00:00,0.0,64.0,16.1,1.6,10.044
1,1,2,2024-07-07 14:04:49.750000+00:00,0.0,63.0,16.2,1.6,47.246
2,1,3,2024-07-07 14:06:21.461000+00:00,0.0,62.0,16.3,2.0,18.957
3,1,4,2024-07-07 14:07:53.675000+00:00,0.0,61.0,16.3,1.9,51.154
4,1,5,2024-07-07 14:09:25.527000+00:00,0.0,60.0,16.2,2.2,22.990
5,1,6,2024-07-07 14:10:57.353000+00:00,0.0,60.0,16.3,2.4,54.814
6,1,7,2024-07-07 14:12:29.113000+00:00,0.0,61.0,16.3,2.1,26.554
7,1,8,2024-07-07 14:14:01.043000+00:00,0.0,60.0,16.4,2.4,58.476
8,1,9,2024-07-07 14:15:33.002000+00:00,0.0,61.0,16.5,2.0,30.422
9,1,10,2024-07-07 14:17:04.948000+00:00,0.0,61.0,16.8,1.5,2.385


In [129]:
model_features = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect",
    "track_evolution",
    "track_evolution_available",
    "rainfall",
    "humidity",
    "air_temperature",
    "wind_speed"
]

target = "lap_duration"

pace_model_data_weather = enhanced_lap_data_weather[
    model_features + [target, "lap_number"]
].copy()

print("Shape:", pace_model_data_weather.shape)

print("\nMissing values:")
display(
    pace_model_data_weather.isna().sum()
)

print("\nColumns:")
print(pace_model_data_weather.columns.tolist())

KeyError: "['track_evolution_available'] not in index"

In [130]:
# Inspect current enhanced dataset columns

print("enhanced_lap_data_weather columns:")
for i, col in enumerate(enhanced_lap_data_weather.columns):
    print(i, col)

print("\nExisting pace_model_data columns:")
for i, col in enumerate(pace_model_data.columns):
    print(i, col)

enhanced_lap_data_weather columns:
0 meeting_key
1 session_key
2 driver_number
3 lap_number
4 date_start
5 duration_sector_1
6 duration_sector_2
7 duration_sector_3
8 i1_speed
9 i2_speed
10 is_pit_out_lap
11 lap_duration
12 segments_sector_1
13 segments_sector_2
14 segments_sector_3
15 st_speed
16 stint_number
17 compound
18 tyre_age
19 fuel_effect
20 position
21 traffic_ahead
22 traffic_behind
23 position_change
24 tyre_degradation_effect
25 track_evolution
26 rainfall
27 humidity
28 air_temperature
29 wind_speed
30 weather_lag_seconds

Existing pace_model_data columns:
0 meeting_key
1 session_key
2 driver_number
3 lap_number
4 date_start
5 duration_sector_1
6 duration_sector_2
7 duration_sector_3
8 i1_speed
9 i2_speed
10 is_pit_out_lap
11 lap_duration
12 segments_sector_1
13 segments_sector_2
14 segments_sector_3
15 st_speed
16 stint_number
17 compound
18 tyre_age
19 fuel_effect
20 position
21 traffic_ahead
22 traffic_behind
23 position_change
24 tyre_degradation_effect
25 track_evol

In [131]:
# Add weather features to the existing pace_model_data

weather_for_model = enhanced_lap_data_weather[
    [
        "driver_number",
        "lap_number",
        "rainfall",
        "humidity",
        "air_temperature",
        "wind_speed",
        "weather_lag_seconds"
    ]
].copy()

pace_model_data_weather = pace_model_data.merge(
    weather_for_model,
    on=["driver_number", "lap_number"],
    how="left",
    validate="one_to_one"
)

print("Original pace_model_data shape:", pace_model_data.shape)
print("Weather-enhanced shape:", pace_model_data_weather.shape)

print("\nWeather missing values:")
display(
    pace_model_data_weather[
        [
            "rainfall",
            "humidity",
            "air_temperature",
            "wind_speed",
            "weather_lag_seconds"
        ]
    ].isna().sum()
)

print("\nTrack evolution availability:")
display(
    pace_model_data_weather["track_evolution_available"]
    .value_counts(dropna=False)
)

Original pace_model_data shape: (668, 27)
Weather-enhanced shape: (668, 32)

Weather missing values:


rainfall               0
humidity               0
air_temperature        0
wind_speed             0
weather_lag_seconds    0
dtype: int64


Track evolution availability:


track_evolution_available
1    574
0     94
Name: count, dtype: int64

In [132]:
print("Final modeling columns:\n")

for i, col in enumerate(pace_model_data_weather.columns):
    print(i, col)

print("\nTarget statistics:")
display(
    pace_model_data_weather["lap_duration"].describe()
)

print("\nLap number range:")
print(
    "Min:", pace_model_data_weather["lap_number"].min()
)
print(
    "Max:", pace_model_data_weather["lap_number"].max()
)

print("\nRows by chronological split:")

train_mask = pace_model_data_weather["lap_number"] < 45
test_mask = pace_model_data_weather["lap_number"] >= 45

print("Train rows:", train_mask.sum())
print("Test rows:", test_mask.sum())

print("\nTrain lap range:")
print(
    pace_model_data_weather.loc[train_mask, "lap_number"].min(),
    "to",
    pace_model_data_weather.loc[train_mask, "lap_number"].max()
)

print("\nTest lap range:")
print(
    pace_model_data_weather.loc[test_mask, "lap_number"].min(),
    "to",
    pace_model_data_weather.loc[test_mask, "lap_number"].max()
)

Final modeling columns:

0 meeting_key
1 session_key
2 driver_number
3 lap_number
4 date_start
5 duration_sector_1
6 duration_sector_2
7 duration_sector_3
8 i1_speed
9 i2_speed
10 is_pit_out_lap
11 lap_duration
12 segments_sector_1
13 segments_sector_2
14 segments_sector_3
15 st_speed
16 stint_number
17 compound
18 tyre_age
19 fuel_effect
20 position
21 traffic_ahead
22 traffic_behind
23 position_change
24 tyre_degradation_effect
25 track_evolution
26 track_evolution_available
27 rainfall
28 humidity
29 air_temperature
30 wind_speed
31 weather_lag_seconds

Target statistics:


count    668.000000
mean      93.058708
std        3.567319
min       88.293000
25%       91.140000
50%       92.444500
75%       93.480250
max      118.710000
Name: lap_duration, dtype: float64


Lap number range:
Min: 1
Max: 52

Rows by chronological split:
Train rows: 535
Test rows: 133

Train lap range:
1 to 44

Test lap range:
45 to 52


In [133]:
print("Existing pace_model_data dtypes:\n")
display(
    pace_model_data.dtypes
)
print("\nCompound values:")
print(
    pace_model_data["compound"].value_counts(dropna=False)
)

print("Existing model feature candidates:")
existing_features = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect",
    "track_evolution",
    "track_evolution_available"
]
print(existing_features)

print("Features present in pace_model_data:")
for feature in existing_features:
    print(f"{feature}: {feature in pace_model_data.columns}")

print("\nFeatures present in pace_model_data_weather:")
for feature in existing_features:
    print(f"{feature}: {feature in pace_model_data_weather.columns}")



Existing pace_model_data dtypes:



meeting_key                                int64
session_key                                int64
driver_number                              int64
lap_number                                 int64
date_start                   datetime64[us, UTC]
duration_sector_1                        float64
duration_sector_2                        float64
duration_sector_3                        float64
i1_speed                                 float64
i2_speed                                 float64
is_pit_out_lap                              bool
lap_duration                             float64
segments_sector_1                         object
segments_sector_2                         object
segments_sector_3                         object
st_speed                                 float64
stint_number                             float64
compound                                     str
tyre_age                                 float64
fuel_effect                              float64
position            


Compound values:
compound
MEDIUM    467
SOFT      161
HARD       40
Name: count, dtype: int64
Existing model feature candidates:
['driver_number', 'compound', 'tyre_age', 'fuel_effect', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution', 'track_evolution_available']
Features present in pace_model_data:
driver_number: True
compound: True
tyre_age: True
fuel_effect: True
traffic_ahead: True
traffic_behind: True
position_change: True
tyre_degradation_effect: True
track_evolution: True
track_evolution_available: True

Features present in pace_model_data_weather:
driver_number: True
compound: True
tyre_age: True
fuel_effect: True
traffic_ahead: True
traffic_behind: True
position_change: True
tyre_degradation_effect: True
track_evolution: True
track_evolution_available: True


In [134]:
# Inspect available DataFrames containing track evolution

print("enhanced_lap_data columns:")
print(enhanced_lap_data.columns.tolist())

print("\npace_model_data columns:")
print(pace_model_data.columns.tolist())

print("\npace_model_data_weather columns:")
print(pace_model_data_weather.columns.tolist())

enhanced_lap_data columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution']

pace_model_data columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution', 'track_evolution_available']

pace_model_data_weather columns:
['meet

In [135]:
print("enhanced_lap_data:")
print(enhanced_lap_data.columns.tolist())

print("\nenhanced_lap_data_weather:")
print(enhanced_lap_data_weather.columns.tolist())

print("\nTrack evolution columns:")
print([
    col for col in enhanced_lap_data.columns
    if "track" in col.lower() or "evolution" in col.lower()
])

print("\nWeather DataFrame track columns:")
print([
    col for col in enhanced_lap_data_weather.columns
    if "track" in col.lower() or "evolution" in col.lower()
])

enhanced_lap_data:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution']

enhanced_lap_data_weather:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution', 'rainfall', 'humidity', 'air_temperature', 'wind_speed', 'weather_lag_secon

In [136]:
# ============================================================
# CLEAN DUPLICATE TRACK EVOLUTION COLUMNS
# ============================================================

# Keep the newly calculated track evolution from race_pace_by_lap
enhanced_lap_data["track_evolution"] = enhanced_lap_data["track_evolution_y"]

# Keep the availability flag
enhanced_lap_data["track_evolution_available"] = (
    enhanced_lap_data["track_evolution_available"].astype(bool)
)

# Remove duplicate columns
enhanced_lap_data = enhanced_lap_data.drop(
    columns=["track_evolution_x", "track_evolution_y"]
)

print("Track evolution successfully restored.")

print(
    enhanced_lap_data[
        [
            "lap_number",
            "track_evolution",
            "track_evolution_available"
        ]
    ].tail(15)
)

print("\nTrack evolution summary:")
display(
    enhanced_lap_data[
        ["track_evolution", "track_evolution_available"]
    ].describe(include="all")
)

KeyError: 'track_evolution_y'

In [137]:
# ============================================================
# REBUILD WEATHER + TRACK EVOLUTION DATASET
# ============================================================

enhanced_lap_data_weather = enhanced_lap_data.merge(
    weather_features,
    on=["driver_number", "lap_number"],
    how="left",
    validate="one_to_one"
)

print("Original shape:", enhanced_lap_data.shape)
print("Weather-enhanced shape:", enhanced_lap_data_weather.shape)

print("\nTrack evolution columns:")
print([
    col for col in enhanced_lap_data_weather.columns
    if "track_evolution" in col
])

print("\nWeather missing values:")
display(
    enhanced_lap_data_weather[
        [
            "rainfall",
            "humidity",
            "air_temperature",
            "wind_speed",
            "weather_lag_seconds"
        ]
    ].isna().sum()
)

Original shape: (962, 26)
Weather-enhanced shape: (962, 31)

Track evolution columns:
['track_evolution']

Weather missing values:


rainfall               1
humidity               1
air_temperature        1
wind_speed             1
weather_lag_seconds    1
dtype: int64

In [138]:
model_features = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect",
    "track_evolution",
    "track_evolution_available",
    "rainfall",
    "humidity",
    "air_temperature",
    "wind_speed"
]

target = "lap_duration" 

pace_model_data_weather = enhanced_lap_data_weather[
    model_features + [target, "lap_number"]
].copy()

print("Shape:", pace_model_data_weather.shape)

print("\nMissing values:")
display(
    pace_model_data_weather.isna().sum()
)

print("\nColumns:")
print(pace_model_data_weather.columns.tolist())

KeyError: "['track_evolution_available'] not in index"

In [139]:
# ============================================================
# CHECK THE PREVIOUS MODEL'S VALIDATION POPULATION
# ============================================================

baseline_features = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect"
]

target = "lap_duration"

baseline_check = pace_model_data[
    baseline_features + [target, "lap_number"]
].copy()

print("Baseline dataset shape:", baseline_check.shape)

print("\nMissing values in baseline features:")
display(
    baseline_check[
        baseline_features + [target]
    ].isna().sum()
)

# Rows that were eligible for the previous model
baseline_valid = baseline_check.dropna(
    subset=baseline_features + [target]
).copy()

print("\nBaseline valid rows:", len(baseline_valid))

print(
    "Baseline valid lap range:",
    baseline_valid["lap_number"].min(),
    "to",
    baseline_valid["lap_number"].max()
)

Baseline dataset shape: (668, 10)

Missing values in baseline features:


driver_number               0
compound                    0
tyre_age                    0
fuel_effect                11
traffic_ahead               0
traffic_behind              0
position_change             0
tyre_degradation_effect    18
lap_duration                0
dtype: int64


Baseline valid rows: 639
Baseline valid lap range: 2 to 52


In [140]:
weather_model_features = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect",
    "track_evolution",
    "rainfall",
    "humidity",
    "air_temperature",
    "wind_speed"
]

weather_model_check = pace_model_data_weather[
    weather_model_features + ["lap_duration", "lap_number"]
].copy()

print("Canddate rows:", len(weather_model_check))
print("\nMissing values:")
display(
    weather_model_check.isna().sum()
)

weather_model_valid = weather_model_check.dropna(
    subset=weather_model_features + ["lap_duration"]
).copy()

print("Valid rows:", len(weather_model_valid))

print(
    "Valid lap range:",
    weather_model_valid["lap_number"].min(),
    "to",
    weather_model_valid["lap_number"].max()
)

Canddate rows: 668

Missing values:


driver_number               0
compound                    0
tyre_age                    0
fuel_effect                11
traffic_ahead               0
traffic_behind              0
position_change             0
tyre_degradation_effect    18
track_evolution             0
rainfall                    0
humidity                    0
air_temperature             0
wind_speed                  0
lap_duration                0
lap_number                  0
dtype: int64

Valid rows: 639
Valid lap range: 2 to 52


In [141]:
comparison_data = weather_model_valid.copy()

train_data = comparison_data[
    comparison_data["lap_number"] < 45
].copy()

test_data = comparison_data[
    comparison_data["lap_number"] >= 45
].copy()
print("Total comparison rows:", len(comparison_data))

print("\nTraining rows:", len(train_data))
print(
    "Training lap range:",
    train_data["lap_number"].min(),
    "to",
    train_data["lap_number"].max()
)

print("\nTest rows:", len(test_data))
print(
    "Test lap range:",
    test_data["lap_number"].min(),
    "to",
    test_data["lap_number"].max()
)

Total comparison rows: 639

Training rows: 506
Training lap range: 2 to 44

Test rows: 133
Test lap range: 45 to 52


In [142]:
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

model_specs = {
    "Baseline": [
        "driver_number",
        "compound",
        "tyre_age",
        "fuel_effect",
        "traffic_ahead",
        "traffic_behind",
        "position_change",
        "tyre_degradation_effect"
    ],

    "With track evolution": [
        "driver_number",
        "compound",
        "tyre_age",
        "fuel_effect",
        "traffic_ahead",
        "traffic_behind",
        "position_change",
        "tyre_degradation_effect",
        "track_evolution"
    ],

    "With track + weather": [
        "driver_number",
        "compound",
        "tyre_age",
        "fuel_effect",
        "traffic_ahead",
        "traffic_behind",
        "position_change",
        "tyre_degradation_effect",
        "track_evolution",
        "rainfall",
        "humidity",
        "air_temperature",
        "wind_speed"
    ]
}

results = []

for model_name, features in model_specs.items():
    X_train = train_data[features].copy()
    X_test = test_data[features].copy()

    y_train = train_data["lap_duration"]
    y_test = test_data["lap_duration"]
    X_train = pd.get_dummies(
        X_train,
        columns=["compound"],
        drop_first=True
    )

    X_test = pd.get_dummies(
        X_test,
        columns=["compound"],
        drop_first=True
    )
    X_test = X_test.reindex(
        columns=X_train.columns,
        fill_value=0
    )
    X_train = X_train.astype(float)
    X_test = X_test.astype(float)

    # Add intercept
    X_train = sm.add_constant(X_train)
    X_test = sm.add_constant(X_test)

    # Fit OLS
    model = sm.OLS(y_train, X_train).fit()

    # Predict
    predictions = model.predict(X_test)

    rmse = np.sqrt(
        mean_squared_error(y_test, predictions)
    )

    r2 = r2_score(y_test, predictions)

    results.append({
        "model": model_name,
        "RMSE": rmse,
        "R2": r2
    })

comparison_results = pd.DataFrame(results)

print(comparison_results.round(4).to_string(index=False))

               model   RMSE      R2
            Baseline 1.6060 -0.5317
With track evolution 1.5127 -0.3588
With track + weather 1.6854 -0.6868


C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\2751063655.py:78: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y_train, X_train).fit()
C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\2751063655.py:78: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y_train, X_train).fit()
C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\2751063655.py:78: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y_train, X_train).fit()


In [143]:
weather_features = [
    "track_evolution",
    "rainfall",
    "humidity",
    "air_temperature",
    "wind_speed"
]

X_train = train_data[
    [
        "driver_number",
        "compound",
        "tyre_age",
        "fuel_effect",
        "traffic_ahead",
        "traffic_behind",
        "position_change",
        "tyre_degradation_effect"
    ] + weather_features
].copy()

y_train = train_data["lap_duration"]
X_train = pd.get_dummies(
    X_train,
    columns=["compound"],
    drop_first=True
)

X_train = X_train.astype(float)
X_train = sm.add_constant(X_train)

weather_ols = sm.OLS(y_train, X_train).fit()

print(weather_ols.summary())

                            OLS Regression Results                            
Dep. Variable:           lap_duration   R-squared:                       0.408
Model:                            OLS   Adj. R-squared:                  0.392
Method:                 Least Squares   F-statistic:                     26.04
Date:                Fri, 02 Oct 2026   Prob (F-statistic):           4.51e-48
Time:                        11:18:20   Log-Likelihood:                -1049.4
No. Observations:                 506   AIC:                             2127.
Df Residuals:                     492   BIC:                             2186.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\1050197140.py:32: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  weather_ols = sm.OLS(y_train, X_train).fit()


In [144]:
weather_regime = lap_weather_aligned[
    [
        "driver_number",
        "lap_number",
        "date_start",
        "rainfall"
    ]
].copy()

weather_regime = weather_regime.sort_values("date_start").reset_index(drop=True)

weather_regime["rainfall_change"] = (
    weather_regime["rainfall"]
    .diff()
    .fillna(0)
)

def classify_regime(row):
    if row["rainfall_change"] != 0:
        return "TRANSITION"
    elif row["rainfall"] == 1:
        return "WET"
    else:
        return "DRY"

weather_regime["race_regime"] = weather_regime.apply(
    classify_regime,
    axis=1
)
print(weather_regime[
    [
        "lap_number",
        "date_start",
        "rainfall",
        "rainfall_change",
        "race_regime"
    ]
].head(20))

print("\nRegime counts:")
print(weather_regime["race_regime"].value_counts())


    lap_number                       date_start  ...  rainfall_change  race_regime
0            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
1            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
2            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
3            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
4            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
5            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
6            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
7            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
8            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
9            1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
10           1 2024-07-07 14:03:12.540000+00:00  ...              0.0          DRY
11  

In [145]:
lap_weather_regime = (
    lap_weather_aligned[
        [
            "lap_number",
            "date_start",
            "rainfall"
        ]
    ]
    .sort_values(["lap_number","date_start"])
    .groupby("lap_number", as_index=False)
    .first()
)

lap_weather_regime["rainfall_change"] = (
    lap_weather_regime["rainfall"]
    .diff()
    .fillna(0)
)
def classify_race_regime(row):
    if row["rainfall_change"] != 0:
        return "TRANSITION"
    elif row["rainfall"] == 1:
        return "WET"
    else:
        return "DRY"

lap_weather_regime["race_regime"] = (
    lap_weather_regime.apply(classify_race_regime, axis=1)
)

print(lap_weather_regime[
    [
        "lap_number",
        "date_start",
        "rainfall",
        "rainfall_change",
        "race_regime"
    ]
].to_string(index=False))

print("\nRegime counts:")
print(lap_weather_regime["race_regime"].value_counts())

print(
    lap_weather_regime[
    [
        "lap_number",
        "rainfall",
        "rainfall_change",
        "race_regime"
    ]
    ].to_string(index=False)
)


 lap_number                       date_start  rainfall  rainfall_change race_regime
          1 2024-07-07 14:03:12.540000+00:00         0              0.0         DRY
          2 2024-07-07 14:04:48.099000+00:00         0              0.0         DRY
          3 2024-07-07 14:06:19.683000+00:00         0              0.0         DRY
          4 2024-07-07 14:07:50.950000+00:00         0              0.0         DRY
          5 2024-07-07 14:09:22.707000+00:00         0              0.0         DRY
          6 2024-07-07 14:10:54.252000+00:00         0              0.0         DRY
          7 2024-07-07 14:12:26.014000+00:00         0              0.0         DRY
          8 2024-07-07 14:13:57.639000+00:00         0              0.0         DRY
          9 2024-07-07 14:15:29.420000+00:00         0              0.0         DRY
         10 2024-07-07 14:17:00.895000+00:00         0              0.0         DRY
         11 2024-07-07 14:18:32.560000+00:00         0              0.0     

In [146]:
lap_weather_regime["weather_state"] = "DRY"

# Currently raining
lap_weather_regime.loc[
    lap_weather_regime["rainfall"] == 1,
    "weather_state"
] = "RAINING"

# Rain has just stopped
lap_weather_regime.loc[
    lap_weather_regime["rainfall_change"] == -1,
    "weather_state"
] = "RAIN_STOPPED"

print(
    lap_weather_regime[
        [
            "lap_number",
            "rainfall",
            "rainfall_change",
            "weather_state"
        ]
    ].to_string(index=False)
)

print("\nWeather-state counts:")
print(lap_weather_regime["weather_state"].value_counts())

 lap_number  rainfall  rainfall_change weather_state
          1         0              0.0           DRY
          2         0              0.0           DRY
          3         0              0.0           DRY
          4         0              0.0           DRY
          5         0              0.0           DRY
          6         0              0.0           DRY
          7         0              0.0           DRY
          8         0              0.0           DRY
          9         0              0.0           DRY
         10         0              0.0           DRY
         11         0              0.0           DRY
         12         0              0.0           DRY
         13         0              0.0           DRY
         14         0              0.0           DRY
         15         0              0.0           DRY
         16         0              0.0           DRY
         17         0              0.0           DRY
         18         0              0.0        

In [147]:
regime_analysis = enhanced_lap_data_weather.merge(
    lap_weather_regime[
        [
            "lap_number",
            "weather_state"
        ]
    ],
    on="lap_number",
    how="left",
    validate="many_to_one"
)

regime_pace_summary = (
    regime_analysis
    .groupby("weather_state")["lap_duration"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max"
    )
    .round(3)
)

print(regime_pace_summary)

               count     mean   median    std     min      max
weather_state                                                 
DRY              512   92.648   92.360  3.052  88.293  125.507
RAINING          411  102.322  102.481  8.086  89.137  128.459
RAIN_STOPPED      36   93.702   90.884  8.008  89.233  116.753


In [148]:
pace_model_data_regime = pace_model_data_weather.merge(
    lap_weather_regime[
        [
            "lap_number",
            "weather_state"
        ]
    ],
    on="lap_number",
    how="left",
    validate="many_to_one"
)

print("Shape:", pace_model_data_regime.shape)

print("\nWeather-state counts:")
print(pace_model_data_regime["weather_state"].value_counts())

print("\nMissing weather states:")
print(pace_model_data_regime["weather_state"].isna().sum())

print("\nColumns:")
print(pace_model_data_regime.columns.tolist())

Shape: (668, 33)

Weather-state counts:
weather_state
DRY             509
RAINING         127
RAIN_STOPPED     32
Name: count, dtype: int64

Missing weather states:
0

Columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age', 'fuel_effect', 'position', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution', 'track_evolution_available', 'rainfall', 'humidity', 'air_temperature', 'wind_speed', 'weather_lag_seconds', 'weather_state']


In [149]:
# ============================================
# OLS: Track evolution vs Track + Weather Regime
# ============================================

import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Features for the controlled experiment
base_features = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect",
]

track_features = base_features + [
    "track_evolution"
]

regime_features = track_features + [
    "weather_state"
]

target = "lap_duration"

# Keep exactly the rows required by both models
experiment_data = pace_model_data_regime[
    track_features + ["weather_state", target, "lap_number"]
].dropna().copy()

# Chronological split
train_data = experiment_data[
    experiment_data["lap_number"] < 45
].copy()

test_data = experiment_data[
    experiment_data["lap_number"] >= 45
].copy()

print("Total rows:", len(experiment_data))
print("Train rows:", len(train_data))
print("Test rows:", len(test_data))

# --------------------------------------------
# Model 1: Track evolution
# --------------------------------------------

X_train_track = pd.get_dummies(
    train_data[track_features],
    columns=["compound"],
    drop_first=True
)

X_test_track = pd.get_dummies(
    test_data[track_features],
    columns=["compound"],
    drop_first=True
)

# Ensure identical columns
X_test_track = X_test_track.reindex(
    columns=X_train_track.columns,
    fill_value=0
)

X_train_track = sm.add_constant(X_train_track)
X_test_track = sm.add_constant(X_test_track)

model_track = sm.OLS(
    train_data[target],
    X_train_track
).fit()

pred_track = model_track.predict(X_test_track)

# --------------------------------------------
# Model 2: Track evolution + weather regime
# --------------------------------------------

X_train_regime = pd.get_dummies(
    train_data[regime_features],
    columns=["compound", "weather_state"],
    drop_first=True
)

X_test_regime = pd.get_dummies(
    test_data[regime_features],
    columns=["compound", "weather_state"],
    drop_first=True
)

# Ensure identical columns
X_test_regime = X_test_regime.reindex(
    columns=X_train_regime.columns,
    fill_value=0
)

X_train_regime = sm.add_constant(X_train_regime)
X_test_regime = sm.add_constant(X_test_regime)

model_regime = sm.OLS(
    train_data[target],
    X_train_regime
).fit()

pred_regime = model_regime.predict(X_test_regime)

# --------------------------------------------
# Compare
# --------------------------------------------

results = pd.DataFrame({
    "model": [
        "With track evolution",
        "With track + weather regime"
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(test_data[target], pred_track)),
        np.sqrt(mean_squared_error(test_data[target], pred_regime))
    ],
    "R2": [
        r2_score(test_data[target], pred_track),
        r2_score(test_data[target], pred_regime)
    ]
})

print("\nModel comparison:")
print(results.round(4))

Total rows: 639
Train rows: 506
Test rows: 133

Model comparison:
                         model    RMSE      R2
0         With track evolution  1.5127 -0.3588
1  With track + weather regime  1.8678 -1.0718


C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\2816196312.py:78: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit()
C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\2816196312.py:110: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit()


In [156]:
df = enhanced_lap_data_weather.copy()

# Remove the single row where rainfall is unknown
df = df.dropna(subset=["rainfall"]).copy()

# Define weather regime
df["weather_regime"] = np.where(
    df["rainfall"] == 1,
    "WET",
    "DRY"
)

print("Weather regime counts:")
print(df["weather_regime"].value_counts())

print("\nWeather regime by rainfall:")
print(
    pd.crosstab(
        df["rainfall"],
        df["weather_regime"]
    )
)

Weather regime counts:
weather_regime
DRY    554
WET    407
Name: count, dtype: int64

Weather regime by rainfall:
weather_regime  DRY  WET
rainfall                
0.0             554    0
1.0               0  407


In [158]:
import numpy as np

enhanced_lap_data_weather = enhanced_lap_data_weather.copy()

# Remove the one row where rainfall is missing
enhanced_lap_data_weather = enhanced_lap_data_weather.dropna(
    subset=["rainfall"]
).copy()

# Create weather regime
enhanced_lap_data_weather["weather_regime"] = np.where(
    enhanced_lap_data_weather["rainfall"] == 1,
    "WET",
    "DRY"
)

print(enhanced_lap_data_weather["weather_regime"].value_counts())

weather_regime
DRY    554
WET    407
Name: count, dtype: int64


In [165]:
print("Features:")
print(features)

print("\nCategorical columns in features:")
for col in features:
    if train[col].dtype == "object":
        print(col, train[col].unique())


Features:
['driver_number', 'compound', 'tyre_age', 'fuel_effect', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution']

Categorical columns in features:


In [172]:
# ============================================================
# SEPARATE DRY / WET OLS MODELS
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score


# ------------------------------------------------------------
# 1. Split training data by weather regime
# ------------------------------------------------------------

dry_train = train[train["weather_regime"] == "DRY"].copy()
wet_train = train[train["weather_regime"] == "WET"].copy()

print("=" * 60)
print("TRAINING DATA")
print("=" * 60)

print("Dry training rows:", len(dry_train))
print("Wet training rows:", len(wet_train))


# ------------------------------------------------------------
# 2. Prepare features
# ------------------------------------------------------------

def prepare_features(data, features, columns=None):

    X = data[features].copy()

    # Encode tyre compound
    X = pd.get_dummies(
        X,
        columns=["compound"],
        drop_first=True,
        dtype=float
    )

    # During prediction, force exactly the same
    # columns that were used during training
    if columns is not None:
        X = X.reindex(
            columns=columns,
            fill_value=0
        )

    return X.astype(float)


# ------------------------------------------------------------
# 3. Train OLS model
# ------------------------------------------------------------

def fit_ols(train_data, features, target):

    X = prepare_features(
        train_data,
        features
    )

    y = train_data[target].astype(float)

    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    return model, X.columns


# ------------------------------------------------------------
# 4. Train separate Dry and Wet models
# ------------------------------------------------------------

dry_model, dry_columns = fit_ols(
    dry_train,
    features,
    target
)

wet_model, wet_columns = fit_ols(
    wet_train,
    features,
    target
)

print("\nOLS MODELS TRAINED")
print("-" * 60)
print("Dry model features:", len(dry_columns))
print("Wet model features:", len(wet_columns))


# ------------------------------------------------------------
# 5. Prepare test data
# ------------------------------------------------------------

test = test.copy()

test["prediction"] = np.nan


# ------------------------------------------------------------
# 6. Predict DRY test laps
# ------------------------------------------------------------

dry_test = test["weather_regime"] == "DRY"

print("\nDRY TEST ROWS:", dry_test.sum())

if dry_test.sum() > 0:

    X_dry_test = prepare_features(
        test.loc[dry_test],
        features,
        columns=dry_columns.drop("const")
    )

    X_dry_test = sm.add_constant(
        X_dry_test,
        has_constant="add"
    )

    test.loc[dry_test, "prediction"] = dry_model.predict(
        X_dry_test
    )


# ------------------------------------------------------------
# 7. Predict WET test laps
# ------------------------------------------------------------

wet_test = test["weather_regime"] == "WET"

print("WET TEST ROWS:", wet_test.sum())

if wet_test.sum() > 0:

    X_wet_test = prepare_features(
        test.loc[wet_test],
        features,
        columns=wet_columns.drop("const")
    )

    X_wet_test = sm.add_constant(
        X_wet_test,
        has_constant="add"
    )

    test.loc[wet_test, "prediction"] = wet_model.predict(
        X_wet_test
    )


# ------------------------------------------------------------
# 8. Prediction check
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PREDICTION CHECK")
print("=" * 60)

print("Total test rows:", len(test))
print(
    "Predictions generated:",
    test["prediction"].notna().sum()
)
print(
    "Missing predictions:",
    test["prediction"].isna().sum()
)


# ------------------------------------------------------------
# 9. Overall evaluation
# ------------------------------------------------------------

valid = test["prediction"].notna()

if valid.sum() > 0:

    overall_rmse = np.sqrt(
        mean_squared_error(
            test.loc[valid, target],
            test.loc[valid, "prediction"]
        )
    )

    overall_r2 = r2_score(
        test.loc[valid, target],
        test.loc[valid, "prediction"]
    )

else:

    overall_rmse = np.nan
    overall_r2 = np.nan


# ------------------------------------------------------------
# 10. Dry evaluation
# ------------------------------------------------------------

dry_valid = (
    (test["weather_regime"] == "DRY")
    & test["prediction"].notna()
)

if dry_valid.sum() > 0:

    dry_rmse = np.sqrt(
        mean_squared_error(
            test.loc[dry_valid, target],
            test.loc[dry_valid, "prediction"]
        )
    )

    dry_r2 = r2_score(
        test.loc[dry_valid, target],
        test.loc[dry_valid, "prediction"]
    )

else:

    dry_rmse = np.nan
    dry_r2 = np.nan


# ------------------------------------------------------------
# 11. Wet evaluation
# ------------------------------------------------------------

wet_valid = (
    (test["weather_regime"] == "WET")
    & test["prediction"].notna()
)

if wet_valid.sum() > 0:

    wet_rmse = np.sqrt(
        mean_squared_error(
            test.loc[wet_valid, target],
            test.loc[wet_valid, "prediction"]
        )
    )

    wet_r2 = r2_score(
        test.loc[wet_valid, target],
        test.loc[wet_valid, "prediction"]
    )

else:

    wet_rmse = np.nan
    wet_r2 = np.nan


# ------------------------------------------------------------
# 12. Results table
# ------------------------------------------------------------

results = pd.DataFrame({
    "model": [
        "Separate Dry/Wet OLS",
        "Dry OLS",
        "Wet OLS"
    ],
    "test_rows": [
        valid.sum(),
        dry_valid.sum(),
        wet_valid.sum()
    ],
    "RMSE": [
        overall_rmse,
        dry_rmse,
        wet_rmse
    ],
    "R2": [
        overall_r2,
        dry_r2,
        wet_r2
    ]
})


# ------------------------------------------------------------
# 13. Final results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SEPARATE WEATHER REGIME OLS RESULTS")
print("=" * 60)

print(results.round(4))

TRAINING DATA
Dry training rows: 349
Wet training rows: 119

OLS MODELS TRAINED
------------------------------------------------------------
Dry model features: 11
Wet model features: 11

DRY TEST ROWS: 95
WET TEST ROWS: 0

PREDICTION CHECK
Total test rows: 95
Predictions generated: 95
Missing predictions: 0

SEPARATE WEATHER REGIME OLS RESULTS
                  model  test_rows    RMSE      R2
0  Separate Dry/Wet OLS         95  1.3251  0.1367
1               Dry OLS         95  1.3251  0.1367
2               Wet OLS          0     NaN     NaN


C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\3495866328.py:68: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y, X).fit()
C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\3495866328.py:68: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y, X).fit()


In [189]:
print(model_df.shape)
print(model_df.head())
print(model_df.tail())
print(model_df.columns.tolist())

print(
    model_df["weather_regime"].value_counts()
)

(563, 11)
    driver_number compound  ...  lap_duration  weather_regime
57             24     SOFT  ...        93.221             DRY
58             18   MEDIUM  ...        92.968             DRY
59             14   MEDIUM  ...        93.192             DRY
60             63   MEDIUM  ...        91.799             DRY
61              2   MEDIUM  ...        93.110             DRY

[5 rows x 11 columns]
     driver_number compound  ...  lap_duration  weather_regime
956             23   MEDIUM  ...        89.718             DRY
957             27     SOFT  ...        90.847             DRY
958              4     SOFT  ...        90.913             DRY
959              2     SOFT  ...        92.088             DRY
960             81   MEDIUM  ...        89.928             DRY

[5 rows x 11 columns]
['driver_number', 'compound', 'tyre_age', 'fuel_effect', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution', 'lap_duration', 'weather_regime']
wea

In [190]:
# Inspect weather regime across the ordered dataset

print("First 100 rows:")
print(model_df["weather_regime"].head(100).value_counts())

print("\nLast 100 rows:")
print(model_df["weather_regime"].tail(100).value_counts())

print("\nWeather regime by index blocks:")

model_df["index_block"] = pd.cut(
    model_df.index,
    bins=10,
    labels=False
)

print(
    pd.crosstab(
        model_df["index_block"],
        model_df["weather_regime"]
    )
)

First 100 rows:
weather_regime
DRY    100
Name: count, dtype: int64

Last 100 rows:
weather_regime
DRY    100
Name: count, dtype: int64

Weather regime by index blocks:
weather_regime  DRY  WET
index_block             
0                91    0
1                89    0
2                89    1
3                13   47
4                 0   46
7                 8    0
8                65   25
9                89    0


In [195]:
from sklearn.model_selection import train_test_split

model_data = model_df.drop(columns=["index_block"], errors="ignore").copy()

train_model, test_model = train_test_split(
    model_data,
    test_size=0.20,
    random_state=42,
    stratify=model_data["weather_regime"]
)

print("Train shape:", train_model.shape)
print("Test shape:", test_model.shape)

print("\nTrain weather:")
print(train_model["weather_regime"].value_counts())

print("\nTest weather:")
print(test_model["weather_regime"].value_counts())

features = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "tyre_degradation_effect",
    "track_evolution"
]
target = "lap_duration"

def prepare_ols_features(data, features, columns=None):
    X = data[features].copy()
    X = pd.get_dummies(
        X,
        columns=["compound"],
        drop_first=True,
        dtype=float
    )

    if columns is not None:
        X = X.reindex(columns=columns,fill_value=0)

    return X.astype(float)

X_train = prepare_ols_features(train_model, features)
X_test = prepare_ols_features(
    test_model,
    features,
    columns=X_train.columns
)

y_train = train_model[target].astype(float)
y_test = test_model[target].astype(float)

X_train = sm.add_constant(X_train)
X_test = sm.add_constant(
    X_test,
    has_constant="add"
)

baseline_ols = sm.OLS(
    y_train,
    X_train
).fit()

baseline_pred = baseline_ols.predict(X_test)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_pred)
)

baseline_r2 = r2_score(
    y_test,
    baseline_pred
)

print("BASELINE OLS")
print("RMSE: ", round(baseline_rmse, 4))
print("R^2: ", round(baseline_r2, 4))


Train shape: (450, 11)
Test shape: (113, 11)

Train weather:
weather_regime
DRY    355
WET     95
Name: count, dtype: int64

Test weather:
weather_regime
DRY    89
WET    24
Name: count, dtype: int64
BASELINE OLS
RMSE:  1.891
R^2:  0.5126


C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\1546546128.py:67: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit()


In [197]:
dry_train = train_model[
    train_model["weather_regime"] == "DRY"
]
wet_train = train_model[
    train_model["weather_regime"] == "WET"
].copy()

dry_test = test_model[
    test_model["weather_regime"] == "DRY"
].copy()

wet_test = test_model[
    test_model["weather_regime"] == "WET"
].copy()

print("Dry train:", len(dry_train))
print("Wet train:", len(wet_train))
print("Dry test:", len(dry_test))
print("Wet test:", len(wet_test))

dry_X = prepare_ols_features(dry_train, features)
wet_X = prepare_ols_features(wet_train, features)

dry_model = sm.OLS(
    dry_train[target].astype(float),
    sm.add_constant(dry_X)
).fit()

wet_model = sm.OLS(
    wet_train[target].astype(float),
    sm.add_constant(wet_X)
).fit()

predictions = np.empty(len(test_model))

dry_mask = test_model["weather_regime"].values == "DRY"
wet_mask = test_model["weather_regime"].values == "WET"

dry_X_test = prepare_ols_features(
    dry_test,
    features,
    columns=dry_X.columns
)
wet_X_test = prepare_ols_features(
    wet_test,
    features,
    columns=wet_X.columns
)

predictions[dry_mask] = dry_model.predict(
    sm.add_constant(dry_X_test, has_constant="add")  
)
predictions[wet_mask] = wet_model.predict(
    sm.add_constant(wet_X_test, has_constant="add")
)

separate_rmse = np.sqrt(
    mean_squared_error(
        test_model[target],
        predictions
    )
)

separate_r2 = r2_score(
    test_model[target],
    predictions
)

print("\nSEPARATE DRY/WET OLS")
print("RMSE:", round(separate_rmse, 4))
print("R²:", round(separate_r2, 4))

Dry train: 355
Wet train: 95
Dry test: 89
Wet test: 24

SEPARATE DRY/WET OLS
RMSE: 2.0375
R²: 0.4342


C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\2265685544.py:27: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit()
C:\Users\Nihan\AppData\Local\Temp\ipykernel_19796\2265685544.py:32: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit()


In [198]:
print("===== DESIGN MATRIX DIAGNOSTIC =====")

for name, data in [
    ("DRY", dry_train),
    ("WET", wet_train)
]:
    X = prepare_ols_features(data, features)
    X = sm.add_constant(X)

    print(f"\n{name}")
    print("Shape:", X.shape)
    print("Rank:", np.linalg.matrix_rank(X))
    print("Columns:", X.shape[1])
    print("Condition number:", np.linalg.cond(X))
    
    print("\nUnique values:")
    for col in X.columns:
        print(f"{col}: {X[col].nunique()}")

print("\n===== DRIVER / WEATHER COVERAGE =====")

print(
    pd.crosstab(
        train_model["driver_number"],
        train_model["weather_regime"]
    )
)

===== DESIGN MATRIX DIAGNOSTIC =====

DRY
Shape: (355, 11)
Rank: 10
Columns: 11
Condition number: 1.1006452146988901e+17

Unique values:
const: 1
driver_number: 19
tyre_age: 17
fuel_effect: 26
traffic_ahead: 2
traffic_behind: 2
position_change: 5
tyre_degradation_effect: 17
track_evolution: 17
compound_MEDIUM: 2
compound_SOFT: 2

WET
Shape: (95, 11)
Rank: 10
Columns: 11
Condition number: 4.978411360596377e+16

Unique values:
const: 1
driver_number: 19
tyre_age: 11
fuel_effect: 10
traffic_ahead: 2
traffic_behind: 2
position_change: 5
tyre_degradation_effect: 11
track_evolution: 8
compound_MEDIUM: 2
compound_SOFT: 2

===== DRIVER / WEATHER COVERAGE =====
weather_regime  DRY  WET
driver_number           
1                22    7
2                19    6
3                20    6
4                16    5
11               16    2
14               17    7
16               21    1
18               18    6
20               20    6
22               19    7
23               21    7
24            

In [200]:
print("===== CORRELATION / DEPENDENCY CHECK =====")

for name, data in [
    ("DRY", dry_train),
    ("WET", wet_train)
]:
    X = prepare_ols_features(data, features)
    X = sm.add_constant(X)

    print(f"\n{name}")

    corr = X.corr()

    for col in X.columns:
        for other in X.columns:
            if col < other and abs(corr.loc[col, other]) > 0.999999:
                print(
                    f"Near-perfect correlation: "
                    f"{col} <-> {other} = {corr.loc[col, other]:.6f}"
                )

print("\n===== DRIVER NUMBER CHECK =====")

print(
    train_model.groupby("driver_number")["lap_duration"]
    .agg(["count", "mean"])
)

===== CORRELATION / DEPENDENCY CHECK =====

DRY
Near-perfect correlation: tyre_age <-> tyre_degradation_effect = 1.000000

WET
Near-perfect correlation: tyre_age <-> tyre_degradation_effect = 1.000000

===== DRIVER NUMBER CHECK =====
               count       mean
driver_number                  
1                 29  91.873586
2                 25  92.608880
3                 26  93.752500
4                 21  91.658381
11                18  92.681778
14                24  92.874375
16                22  91.663273
18                24  92.733667
20                26  93.027692
22                26  92.968231
23                28  92.597500
24                21  93.460381
27                26  92.215923
31                21  93.945190
44                25  91.414880
55                27  92.017444
63                16  92.423750
77                24  94.184042
81                21  91.335810


In [203]:
features_ols = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "track_evolution"
]

def prepare_ols_features(data, features, columns=None):
    X = data[features].copy()

    X = pd.get_dummies(
        X,
        columns=["driver_number", "compound"],
        drop_first=True,
        dtype=float
    )

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0)

    return X.astype(float)

dry_X = prepare_ols_features(dry_train, features_ols)
wet_X = prepare_ols_features(wet_train, features_ols)

dry_model = sm.OLS(
    dry_train["lap_duration"].astype(float),
    sm.add_constant(dry_X)
).fit()

wet_model = sm.OLS(
    wet_train["lap_duration"].astype(float),
    sm.add_constant(wet_X)
).fit()

print("DRY")
print("Shape:", dry_X.shape)
print("Rank:", np.linalg.matrix_rank(sm.add_constant(dry_X)))
print("Columns:", sm.add_constant(dry_X).shape[1])

print("\nWET")
print("Shape:", wet_X.shape)
print("Rank:", np.linalg.matrix_rank(sm.add_constant(wet_X)))
print("Columns:", sm.add_constant(wet_X).shape[1])

# Prepare test features using the training columns

dry_X_test = prepare_ols_features(
    dry_test,
    features_ols,
    columns=dry_X.columns
)

wet_X_test = prepare_ols_features(
    wet_test,
    features_ols,
    columns=wet_X.columns
)

# Predictions

predictions = np.empty(len(test_model))

dry_mask = test_model["weather_regime"].values == "DRY"
wet_mask = test_model["weather_regime"].values == "WET"

predictions[dry_mask] = dry_model.predict(
    sm.add_constant(dry_X_test, has_constant="add")
)

predictions[wet_mask] = wet_model.predict(
    sm.add_constant(wet_X_test, has_constant="add")
)

# Evaluation

ols_rmse = np.sqrt(
    mean_squared_error(
        test_model["lap_duration"],
        predictions
    )
)

ols_r2 = r2_score(
    test_model["lap_duration"],
    predictions
)

print("===== SEPARATE DRY/WET OLS =====")
print("RMSE:", round(ols_rmse, 4))
print("R²:", round(ols_r2, 4))

DRY
Shape: (355, 26)
Rank: 27
Columns: 27

WET
Shape: (95, 26)
Rank: 27
Columns: 27
===== SEPARATE DRY/WET OLS =====
RMSE: 1.9958
R²: 0.4571


In [204]:
print("\n===== REGIME-SPECIFIC PERFORMANCE =====")

dry_rmse = np.sqrt(
    mean_squared_error(
        dry_test["lap_duration"],
        predictions[dry_mask]
    )
)

wet_rmse = np.sqrt(
    mean_squared_error(
        wet_test["lap_duration"],
        predictions[wet_mask]
    )
)

dry_r2 = r2_score(
    dry_test["lap_duration"],
    predictions[dry_mask]
)

wet_r2 = r2_score(
    wet_test["lap_duration"],
    predictions[wet_mask]
)

print(f"DRY: RMSE = {dry_rmse:.4f}, R² = {dry_r2:.4f}")
print(f"WET: RMSE = {wet_rmse:.4f}, R² = {wet_r2:.4f}")


===== REGIME-SPECIFIC PERFORMANCE =====
DRY: RMSE = 0.8132, R² = 0.7866
WET: RMSE = 4.0375, R² = 0.0173


In [207]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

features_ml = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "track_evolution"
]

target = "lap_duration"

X_train = pd.get_dummies(
    train_model[features_ml],
    columns=["driver_number", "compound"],
    drop_first=True,
    dtype=float
)

X_test = pd.get_dummies(
    test_model[features_ml],
    columns=["driver_number", "compound"],
    drop_first=True,
    dtype=float
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

y_train = train_model[target].astype(float)
y_test = test_model[target].astype(float)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

results_ml = []
predictions_ml = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    rmse = np.sqrt(
        mean_squared_error(y_test, predictions)
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results_ml.append({
        "model": name,
        "RMSE": rmse,
        "R2": r2
    })

    predictions_ml[name] = predictions

ml_comparison = pd.DataFrame(results_ml)
print("\n===== ML MODEL COMPARISON =====")
print(
    ml_comparison
    .sort_values("RMSE")
    .reset_index(drop=True)
)

Training shape: (450, 26)
Testing shape: (113, 26)

===== ML MODEL COMPARISON =====
               model      RMSE        R2
0            XGBoost  0.978384  0.869529
1      Random Forest  1.027447  0.856116
2  Linear Regression  1.782198  0.567081


In [212]:
print("enhanced_lap_data_weather:")
print(enhanced_lap_data_weather.shape)

print("\nWeather columns:")
print([
    col for col in [
        "rainfall",
        "humidity",
        "air_temperature",
        "wind_speed",
        "weather_lag_seconds"
    ]
    if col in enhanced_lap_data_weather.columns
])

print("\nmodel_df columns:")
print(model_df.columns.tolist())

print("\ntrain_model columns:")
print(train_model.columns.tolist())

print("model_df index_block range:")
print(model_df["index_block"].head(10).tolist())

print("\nWeather dataframe index range:")
print(enhanced_lap_data_weather.index[:10].tolist())

print("\nindex_block unique:", model_df["index_block"].nunique())
print("model_df rows:", len(model_df))

print("\nDoes index_block exist in weather dataframe index?")
print(
    model_df["index_block"].isin(enhanced_lap_data_weather.index).sum(),
    "of",
    len(model_df)
)

print("weather_for_model:")
print(weather_for_model.shape)
print(weather_for_model.columns.tolist())

print("\npace_model_data_weather:")
print(pace_model_data_weather.shape)
print(pace_model_data_weather.columns.tolist())

print("\ncomparison_data:")
print(comparison_data.shape)
print(comparison_data.columns.tolist())

enhanced_lap_data_weather:
(961, 32)

Weather columns:
['rainfall', 'humidity', 'air_temperature', 'wind_speed', 'weather_lag_seconds']

model_df columns:
['driver_number', 'compound', 'tyre_age', 'fuel_effect', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution', 'lap_duration', 'weather_regime', 'index_block']

train_model columns:
['driver_number', 'compound', 'tyre_age', 'fuel_effect', 'traffic_ahead', 'traffic_behind', 'position_change', 'tyre_degradation_effect', 'track_evolution', 'lap_duration', 'weather_regime']
model_df index_block range:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Weather dataframe index range:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

index_block unique: 8
model_df rows: 563

Does index_block exist in weather dataframe index?
563 of 563
weather_for_model:
(962, 7)
['driver_number', 'lap_number', 'rainfall', 'humidity', 'air_temperature', 'wind_speed', 'weather_lag_seconds']

pace_model_data_weather:
(668, 32)
['meeting_key', 'sessio

In [214]:
# Start from the weather-enriched dataset
weather_model_df = comparison_data.copy()

# Add weather regime from rainfall
weather_model_df["weather_regime"] = np.where(
    weather_model_df["rainfall"] == 1,
    "WET",
    "DRY"
)

# Remove rows with missing values in the variables we need
weather_model_df = weather_model_df.dropna().reset_index(drop=True)

print("Weather model shape:", weather_model_df.shape)
print("\nWeather regime:")
print(weather_model_df["weather_regime"].value_counts())

from sklearn.model_selection import train_test_split

train_weather, test_weather = train_test_split(
    weather_model_df,
    test_size=0.20,
    random_state=42,
    stratify=weather_model_df["weather_regime"]
)

print("Train:", train_weather.shape)
print("Test:", test_weather.shape)

print("\nTrain weather:")
print(train_weather["weather_regime"].value_counts())

print("\nTest weather:")
print(test_weather["weather_regime"].value_counts())

Weather model shape: (639, 16)

Weather regime:
weather_regime
DRY    520
WET    119
Name: count, dtype: int64
Train: (511, 16)
Test: (128, 16)

Train weather:
weather_regime
DRY    416
WET     95
Name: count, dtype: int64

Test weather:
weather_regime
DRY    104
WET     24
Name: count, dtype: int64


In [216]:
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import numpy as np
import pandas as pd

base_features = [
    "driver_number",
    "compound",
    "tyre_age",
    "fuel_effect",
    "traffic_ahead",
    "traffic_behind",
    "position_change",
    "track_evolution"
]

weather_features = [
    "rainfall",
    "humidity",
    "air_temperature",
    "wind_speed",
]

target = "lap_duration"


def prepare_features(train_df, test_df, features):
    X_train = pd.get_dummies(
        train_df[features],
        columns=["driver_number", "compound"],
        drop_first=True,
        dtype=float
    )

    X_test = pd.get_dummies(
        test_df[features],
        columns=["driver_number", "compound"],
        drop_first=True,
        dtype=float
    )

    X_test = X_test.reindex(
        columns=X_train.columns,
        fill_value=0
    )

    return X_train, X_test


# -------------------------
# BASE MODEL
# -------------------------

X_train_base, X_test_base = prepare_features(
    train_weather,
    test_weather,
    base_features
)

y_train = train_weather[target].astype(float)
y_test = test_weather[target].astype(float)

base_model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

base_model.fit(X_train_base, y_train)

base_pred = base_model.predict(X_test_base)

base_rmse = np.sqrt(mean_squared_error(y_test, base_pred))
base_r2 = r2_score(y_test, base_pred)


# -------------------------
# WEATHER MODEL
# -------------------------

all_features = base_features + weather_features

X_train_weather, X_test_weather = prepare_features(
    train_weather,
    test_weather,
    all_features
)

weather_model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

weather_model.fit(X_train_weather, y_train)

weather_pred = weather_model.predict(X_test_weather)

weather_rmse = np.sqrt(mean_squared_error(y_test, weather_pred))
weather_r2 = r2_score(y_test, weather_pred)


# -------------------------
# COMPARISON
# -------------------------

weather_comparison = pd.DataFrame({
    "model": [
        "Base XGBoost",
        "Base + Weather XGBoost"
    ],
    "RMSE": [
        base_rmse,
        weather_rmse
    ],
    "R2": [
        base_r2,
        weather_r2
    ]
})

print(weather_comparison)

                    model      RMSE        R2
0            Base XGBoost  0.649877  0.917552
1  Base + Weather XGBoost  0.621171  0.924675


In [218]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path.cwd().parent
MODEL_PATH = PROJECT_ROOT / "models"

MODEL_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Models folder:", MODEL_PATH)
print("Exists:", MODEL_PATH.exists())

Project root: c:\Users\Nihan\.vscode\pitwall
Models folder: c:\Users\Nihan\.vscode\pitwall\models
Exists: True


In [221]:
from pathlib import Path
import joblib

MODEL_PATH = Path("../models")
MODEL_PATH.mkdir(exist_ok=True)

joblib.dump(weather_model, MODEL_PATH / "pace_xgb_weather.joblib")
joblib.dump(
    X_train_weather.columns.tolist(),
    MODEL_PATH / "pace_xgb_weather_features.joblib"
)

print("Model saved.")
print("Features:", X_train_weather.columns.tolist())

Model saved.
Features: ['tyre_age', 'fuel_effect', 'traffic_ahead', 'traffic_behind', 'position_change', 'track_evolution', 'rainfall', 'humidity', 'air_temperature', 'wind_speed', 'driver_number_2', 'driver_number_3', 'driver_number_4', 'driver_number_11', 'driver_number_14', 'driver_number_16', 'driver_number_18', 'driver_number_20', 'driver_number_22', 'driver_number_23', 'driver_number_24', 'driver_number_27', 'driver_number_31', 'driver_number_44', 'driver_number_55', 'driver_number_63', 'driver_number_77', 'driver_number_81', 'compound_MEDIUM', 'compound_SOFT']
